# Neighbouring weed species influence the root microbiota and growth of wheat — figures and Python-side tables

Run `Paper_analyses.qmd` first as this notebook reads its outputs from `results/`.



## Setup

In [ ]:
import os, sys, shutil, subprocess, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CODE_DIR   = os.path.abspath(os.environ.get("PAPER_CODE", os.getcwd()))
assert os.path.exists(os.path.join(CODE_DIR, "pooled_check.R")), f"code folder not found ({CODE_DIR}): open the notebook from code"
os.environ["PAPER_CODE"] = CODE_DIR      # read by control_size.run()
PROJECT    = os.path.dirname(CODE_DIR)
DATA_DIR   = os.environ.get("PAPER_DATA", os.path.join(PROJECT, "data"))
RES_DIR    = os.path.join(DATA_DIR, "results_v4")
MATRIX_DIR = os.path.join(DATA_DIR, "matrice_phase1", "df")

# --- THE SWITCH (same as in Paper_analyses.qmd)
RUN_FROM_ASV = False   # True: use the results the qmd regenerated from the ASV tables (run the qmd with the switch first)
# PAPER_FROM_ASV=true before starting Jupyter
FROM_ASV   = RUN_FROM_ASV or os.environ.get("PAPER_FROM_ASV") == "true"
if FROM_ASV:   # DA exports and NST tables regenerated by the qmd (results_from_asv/da_exports, results_from_asv/NST)
    os.environ["PAPER_DA"] = "regenerated"; os.environ["PAPER_NST"] = "qmd"
RESULTS    = os.environ.get("PAPER_RESULTS", os.path.join(PROJECT, "results_from_asv" if FROM_ASV else "results"))   
OUT_DIR    = os.environ.get("PAPER_OUT", RESULTS)                                  
print("code:", CODE_DIR, "\ndata:", DATA_DIR, "\nresults:", RESULTS, "(from the ASV tables)" if FROM_ASV else "")
FIG, TAB   = os.path.join(OUT_DIR, "figures"), os.path.join(OUT_DIR, "tables")
SUPP       = os.path.join(RESULTS, "supporting")
REV        = os.path.join(SUPP, "revision")           
TRAITS     = os.path.join(SUPP, "trait_models")        
for d in (FIG, TAB): os.makedirs(d, exist_ok=True)
DA_ROOTS   = [os.path.join(RESULTS, "da_exports")] if os.environ.get("PAPER_DA") == "regenerated" else [os.path.join(DATA_DIR, "da_exports_paper")]
# nst-generate); "original" = data/NST; "qmd" = the tables the qmd wrote in this run (results/NST).
NST_DIR    = {"regenerated": os.path.join(DATA_DIR, "NST_regenerated"), "original": os.path.join(DATA_DIR, "NST"),
              "qmd": os.path.join(RESULTS, "NST")}[os.environ.get("PAPER_NST", "regenerated")]
RSCRIPT    = os.environ.get("RSCRIPT", shutil.which("Rscript") or "Rscript")
NEEDED = {"Table1_variance_partition.csv": "table-1 (3.1.1)",
          "supporting/revision/fig4_direct_test.csv": "revision-analyses (3.1.1)",
          "da_exports/edgeR_species": "edger-vs-wheat-x-wheat (3.1.2)",
          "Table_S6_biomass_ancova.csv": "table-s5-biomass (3.2.1)",
          "Table_S7_TITAN2_indicators.csv": "table-s7-titan2 (3.2.2)",   
          "supporting/trait_models/trait_joint.csv": "table-2 (3.3)"}
missing = {f: c for f, c in NEEDED.items() if not os.path.exists(os.path.join(RESULTS, f))}
assert not missing, ("render or run Paper_analyses.qmd first; it writes to " + RESULTS + ". Missing (file: qmd chunk): "
                     + "; ".join(f"{f}: {c}" for f, c in missing.items()))   

assert os.path.isdir(DA_ROOTS[0]) and os.path.isdir(NST_DIR), (DA_ROOTS, NST_DIR)                       
import types
from IPython.core.magic import register_cell_magic

@register_cell_magic
def module(line, cell):
    """%%module <name> [alias]: run the cell as the module <name>, as `import <name>` would, and bind it to <alias>."""
    name, *alias = line.split()
    m = types.ModuleType(name); m.__file__ = os.path.join(CODE_DIR, name + ".py")  
    sys.modules[name] = m                               
    exec(compile(cell, f"<module {name}>", "exec"), m.__dict__)
    get_ipython().user_ns[alias[0] if alias else name] = m

code: /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/zenodo/github_repo/code 
data: /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/zenodo/github_repo/data 
results: /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/zenodo/github_repo/results 


AssertionError: render or run Paper_analyses.qmd first; it writes to /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/zenodo/github_repo/results. Missing (file: qmd chunk): Table1_variance_partition.csv: table-1 (3.1.1); supporting/revision/fig4_direct_test.csv: revision-analyses (3.1.1); da_exports/edgeR_species: edger-vs-wheat-x-wheat (3.1.2); Table_S6_biomass_ancova.csv: table-s5-biomass (3.2.1); Table_S7_TITAN2_indicators.csv: table-s7-titan2 (3.2.2); supporting/trait_models/trait_joint.csv: table-2 (3.3)

## Code modules (formerly the `.py` files of the code folder) 

| Module | Alias | Provides | Used in |
|---|---|---|---|
| `si_figures` | `sf` | Supporting Information figures: Fig. S6 (Glomeromycota), Fig. S7 (bacterial assembly), turnover table, family palette | 3.1.1; Fig. 1 and Fig. 2 palettes |
| `main_figures` | `mf` | main-text figures: Fig. 1, Fig. 2, Fig. 3, Fig. 4 | 3.1.1, 3.1.2, 3.2.1, 3.2.2 |
| `da_consensus` | `dc` | differential-abundance helpers: reading the DESeq2 / edgeR / metagenomeSeq exports| 3.1.2 |
| `bacteria_dotplot_revised` | `bdp` | bacterial panel of the genus-level map of the candidate ASVs (Table S3 workbook) | 3.1.2 |
| `fungi_dotplot_cell` | `fdp` | fungal panel of the same map | 3.1.2 |
| `biomass_tests` | `bt` | wheat-biomass tests: ANOVA, scale check, between-soil tests, legume test, Table S4 | 3.2.1 |
| `control_size` | `csz` | control-size check: runs `pooled_check.R` and `vs_alone.R`, classifies the Fig. 2 calls, Table S3 sheets | 3.1.2 |
| `fig_alpha_diversity` | `fa` | α-diversity table and Fig. S5 | 3.1.1 |


### `si_figures` (`sf`)
Supporting Information figures: Fig. S6 (Glomeromycota), Fig. S7 (bacterial assembly), turnover table, family palette. Used in 3.1.1; Fig. 1 and Fig. 2 palettes.

In [ ]:
%%module si_figures sf
# -*- coding: utf-8 -*-
"""Supporting Information figures (Figs S8, S9, S11, S12, S14, S16) and the ASV panel used by Fig. 2.
Called from Paper_figures.ipynb. Soil colours: cereal-dominated (AF193) blue, mixed-system (AF190) orange."""
import os
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

SOILCODE = {"resistant": "Cereal-dominated (AF193)", "Permissive": "Mixed-system (AF190)"}
SOILCOL = {"Cereal-dominated (AF193)": "#3B75AF", "Mixed-system (AF190)": "#E08A2C"}
LIGHT = {"Cereal-dominated (AF193)": "#A9C4E2", "Mixed-system (AF190)": "#F3C994"}
CTRL = ["wheat x wheat", "wheat alone"]
RC = {"font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
      "legend.fontsize": 6, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
      "font.family": "sans-serif", "pdf.fonttype": 42, "svg.fonttype": "none"}
FAMCOL = {"Apiaceae": "#d62728", "Asteraceae": "#1f77b4", "Boraginaceae": "#bcbd22", "Brassicaceae": "#2ca02c",
          "Campanulaceae": "#e377c2", "Caryophyllaceae": "#98df8a", "Fabaceae": "#7f7f7f", "Geraniaceae": "#9467bd",
          "Juncaceae": "#8c564b", "Papaveraceae": "#17becf", "Plantaginaceae": "#ff9896", "Poaceae": "#ff7f0e",
          "Polygonaceae": "#ffbb78", "Ranunculaceae": "#aec7e8", "Wheat": "#000000"}
# old labels in the NST exports
NST_SYN = {"Centaurea cyanus": "Cyanus segetum", "Bromus sterilis": "Anisantha sterilis", "Festuca arundinacea": "Lolium arundinaceum",
           "Festuca myuros": "Lolium arundinaceum", "Viola arvensis": "Vulpia myuros"}


def _save(fig, out):
    for ext in ("png", "pdf", "svg"): fig.savefig(f"{out}.{ext}", dpi=400, bbox_inches="tight")
    plt.close(fig)
    return out + ".png"


def _meta(DF, kg="bacteria"):
    n = 1314 if kg == "bacteria" else 647
    return pd.read_csv(os.path.join(DF, f"metadata_{kg}_{n}ASVs.csv"))


def species_order(meta):
    f = meta[~meta.species.isin(CTRL)].groupby("species").family.first()
    return list(f.reset_index().sort_values(["family", "species"]).species), f.to_dict()


def beta_pair(PA):
    A = PA.astype(np.int32); a = A @ A.T; s = A.sum(1); b = s[:, None] - a; c = s[None, :] - a
    mn = np.minimum(b, c)
    with np.errstate(invalid="ignore", divide="ignore"):
        sor = (b + c) / (2 * a + b + c); sim = mn / (a + mn)
    iu = np.triu_indices(len(A), 1)
    return np.nanmean(sim[iu]), np.nanmean(sor[iu] - sim[iu]), np.nanmean(sor[iu])


def turnover_table(DF):
    rows = []
    for kg, n in (("bacteria", 1314), ("fungi", 647)):
        X = pd.read_csv(os.path.join(DF, f"otu_{kg}_table_R_{n}ASVs.csv"), sep="\t", index_col=0).T.astype(float)
        md = _meta(DF, kg).set_index("sample"); md = md.loc[md.index.intersection(X.index)]
        for code, sl in SOILCODE.items():
            m = md[md.soil == code]; PA = (X.loc[m.index] > 0).values
            t, ne, tot = beta_pair(PA); rows.append(dict(kingdom=kg, soil=sl, group="All pots", n=len(m), turnover=t, nestedness=ne, total=tot))
            grp = m.species.map({"wheat x wheat": "Wheat × wheat", "wheat alone": "Wheat alone"}).fillna(m.family)
            for g in grp.unique():
                idx = (grp == g).values
                if idx.sum() < 3: continue
                t, ne, tot = beta_pair(PA[idx]); rows.append(dict(kingdom=kg, soil=sl, group=g, n=int(idx.sum()), turnover=t, nestedness=ne, total=tot))
    d = pd.DataFrame(rows); d["pct_turnover"] = 100 * d.turnover / d.total
    return d


def fig_turnover(DF, out):
    d = turnover_table(DF); d.to_csv(out + "_values.csv", index=False)
    fams = sorted(set(d.group) - {"All pots", "Wheat × wheat", "Wheat alone"})
    order = ["All pots"] + fams + ["Wheat alone", "Wheat × wheat"]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 4.4), sharey=True, gridspec_kw=dict(wspace=0.08))
        for ax, kg in zip(axes, ("bacteria", "fungi")):
            g = d[d.kingdom == kg].set_index(["group", "soil"])
            for i, grp in enumerate(order):
                for k, s in enumerate(SOILCOL):
                    if (grp, s) not in g.index: continue
                    r = g.loc[(grp, s)]; y = i + (k - 0.5) * 0.38
                    ax.barh(y, r.turnover, height=0.36, color=SOILCOL[s], lw=0)
                    ax.barh(y, r.nestedness, left=r.turnover, height=0.36, color=LIGHT[s], lw=0)
                    ax.text(r.total + 0.01, y, f"{r.pct_turnover:.0f}%", va="center", fontsize=5.5, color="#333333")
            ax.axhline(0.5, color="#999999", lw=0.5); ax.axhline(len(order) - 2.5, color="#999999", lw=0.5)
            ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
            ax.set_xlim(0, 0.95); ax.set_xlabel("Mean pairwise Sørensen dissimilarity")
            ax.set_title(("Bacteria (rpoB)" if kg == "bacteria" else "Fungi (ITS1)"), loc="left", fontweight="bold")
        axes[0].invert_yaxis()
        h = [Patch(color=SOILCOL[s], label=f"{s}: turnover") for s in SOILCOL] + \
            [Patch(facecolor=LIGHT[s], label=f"{s}: nestedness") for s in SOILCOL]
        fig.legend(handles=h, frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.09))
        fig.text(0.99, 0.005, "labels: turnover as % of total dissimilarity", ha="right", fontsize=5.5, color="#555555")
        return _save(fig, out), d


def nst_table(NST_DIR):
    t = []
    for kg, f1, f2 in (("bacteria", "NST_bacteria.csv", "pNST_bacteria.csv"), ("fungi", "NST_fungi.csv", "pnst_fungi.csv")):
        a = pd.read_csv(os.path.join(NST_DIR, f1)); b = pd.read_csv(os.path.join(NST_DIR, f2))
        b["Species"] = b.Species.str.replace("_", " ").str.replace("bursa pastoris", "bursa-pastoris").str.replace("speculum veneris", "speculum-veneris").str.replace("foenum graecum", "foenum-graecum")
        m = a[["Soil", "Species", "NST"]].merge(b[["Soil", "Species", "pNST"]], on=["Soil", "Species"])
        m["Species"] = m.Species.replace(NST_SYN); m["kingdom"] = kg; m["soil"] = m.Soil.map(SOILCODE); t.append(m)
    return pd.concat(t, ignore_index=True)


def fig_nst(NST_DIR, DF, out):
    d = nst_table(NST_DIR); sp, fam = species_order(_meta(DF))
    dropped = sorted(set(d.Species) - set(sp) - set(CTRL)); d = d[d.Species.isin(sp + CTRL)]
    d.to_csv(out + "_values.csv", index=False)
    order = sp + CTRL; lab = [s if s not in CTRL else ("wheat × wheat" if s == "wheat x wheat" else "wheat alone") for s in order]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(1, 4, figsize=(7.2, 4.8), sharey=True, gridspec_kw=dict(wspace=0.14))
        for ax, (kg, col, ttl) in zip(axes, (("bacteria", "NST", "Bacteria, taxonomic NST"), ("bacteria", "pNST", "Bacteria, phylogenetic NST"),
                                             ("fungi", "NST", "Fungi, taxonomic NST"), ("fungi", "pNST", "Fungi, phylogenetic NST"))):
            g = d[d.kingdom == kg]
            for k, s in enumerate(SOILCOL):
                gs = g[g.soil == s].set_index("Species")[col].reindex(order)
                ax.scatter(gs.values, np.arange(len(order)) + (k - 0.5) * 0.3, s=9, color=SOILCOL[s], lw=0, label=s, zorder=3)
            ax.axvline(0.5, color="black", lw=0.6, ls="--")
            ax.set_xlim(-0.02, 1.02); ax.set_xticks([0, 0.5, 1]); ax.set_xticklabels(["0", "0.5", "1"]); ax.set_title(ttl, loc="left"); ax.set_xlabel("NST")
            for j in range(1, len(order)):
                fa = fam.get(order[j], "Wheat"); fb = fam.get(order[j - 1], "Wheat")
                if fa != fb: ax.axhline(j - 0.5, color="#e5e5e5", lw=0.5, zorder=0)
            ax.text(0.25, -1.2, "deterministic", ha="center", fontsize=5.5, color="#555555"); ax.text(0.75, -1.2, "stochastic", ha="center", fontsize=5.5, color="#555555")
        axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(lab, style="italic"); axes[0].invert_yaxis()
        for t_, s in zip(axes[0].get_yticklabels(), order):
            t_.set_color(FAMCOL.get(fam.get(s, "Wheat"), "black")) if False else None
        fig.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=s, markersize=4) for s, c in SOILCOL.items()],
                   frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.05))
        return _save(fig, out), dropped


def fig_rci_family(DF, out):
    meta = _meta(DF).dropna(subset=["Dry_wheat_mass"]).copy(); meta["soilL"] = meta.soil.map(SOILCODE)
    alone = meta[meta.species == "wheat alone"].groupby("soilL").Dry_wheat_mass.mean()
    meta["RCI"] = (meta.soilL.map(alone) - meta.Dry_wheat_mass) / meta.soilL.map(alone)
    sp, fam = species_order(meta); order = sp + ["wheat x wheat"]
    meta[meta.species != "wheat alone"][["soilL", "species", "RCI"]].to_csv(out + "_values.csv", index=False)
    with plt.rc_context(RC):
        fig, axes = plt.subplots(2, 1, figsize=(7.2, 5.4), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.12))
        for ax, s in zip(axes, SOILCOL):
            mu = meta[(meta.soilL == s) & ~meta.species.isin(CTRL)].RCI.mean()
            for i, x in enumerate(order):
                v = meta.loc[(meta.soilL == s) & (meta.species == x), "RCI"].dropna().values
                c = FAMCOL[fam.get(x, "Wheat")]
                bp = ax.boxplot(v, positions=[i], widths=0.65, patch_artist=True, showfliers=True,
                                flierprops=dict(marker="o", markersize=1.8, markerfacecolor="black", markeredgecolor="none"),
                                medianprops=dict(color="black", lw=0.8), whiskerprops=dict(lw=0.6), capprops=dict(lw=0.6))
                bp["boxes"][0].set(facecolor=c, edgecolor="black", lw=0.5)
            ax.axhline(mu, color="black", lw=0.7, ls="--"); ax.axhline(0, color="#777777", lw=0.5, ls=":")
            ax.set_ylabel("Relative Competition Index (RCI)"); ax.set_title(s, loc="left", fontweight="bold", color=SOILCOL[s])
            ax.set_title(f"dashed line: mean RCI of neighbour pots ({mu:.2f}); dotted: RCI = 0 (wheat alone)", loc="right", fontsize=5.5)
        axes[1].set_xticks(range(len(order)))
        axes[1].set_xticklabels([("wheat × wheat" if o == "wheat x wheat" else o) for o in order], rotation=80, style="italic")
        hs = [Patch(facecolor=FAMCOL[f], edgecolor="black", lw=0.4, label=("Wheat (wheat × wheat)" if f == "Wheat" else f))
              for f in list(dict.fromkeys([fam[s] for s in sp])) + ["Wheat"]]
        fig.legend(handles=hs, title="Family", frameon=False, loc="center left", bbox_to_anchor=(0.91, 0.5))
        return _save(fig, out)


def fig_fdr_asv(P, lfc, T, fam, out):
    """P: dict (kingdom, soil) -> set of (neighbour, ASV); lfc: dict (kingdom, soil, neighbour, ASV) -> DESeq2 log2FC."""
    rows = [dict(kingdom=kg, soil=SOILCODE["resistant" if s == "Resistant" else "Permissive"], neighbour=w, ASV=c,
                 l=lfc.get((kg, s, w, c), np.nan)) for (kg, s), pp in P.items() for w, c in pp]
    d = pd.DataFrame(rows)
    d["genus"] = [T[k]["tax"].Genus.get(a) for k, a in zip(d.kingdom, d.ASV)]
    d["phylum"] = [T[k]["tax"].Phylum.get(a) for k, a in zip(d.kingdom, d.ASV)]
    d.to_csv(out + "_values.csv", index=False)
    nb = sorted(fam, key=lambda s: (fam[s], s))
    asv = d.groupby(["kingdom", "phylum", "genus", "ASV"]).neighbour.nunique().reset_index().sort_values(["kingdom", "phylum", "genus", "neighbour"], ascending=[True, True, True, False])
    lab = [f"{g} {a.replace('Cluster_', 'ASV_')}" for g, a in zip(asv.genus, asv.ASV)]
    ypos = {(k, a): i for i, (k, a) in enumerate(zip(asv.kingdom, asv.ASV))}
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(7.2, 0.16 * len(asv) + 2.0))
        for k, s in enumerate(SOILCOL):
            g = d[d.soil == s]
            ax.scatter([nb.index(w) + (k - 0.5) * 0.32 for w in g.neighbour], [ypos[(a, b)] for a, b in zip(g.kingdom, g.ASV)],
                       s=14, color=SOILCOL[s], lw=0, label=s, zorder=3)
        for j in range(1, len(nb)):
            if fam[nb[j]] != fam[nb[j - 1]]: ax.axvline(j - 0.5, color="#dddddd", lw=0.5, zorder=0)
        kb = (asv.kingdom == "bacteria").sum(); ax.axhline(kb - 0.5, color="black", lw=0.6)
        ax.set_yticks(range(len(asv))); ax.set_yticklabels(lab, style="italic"); ax.invert_yaxis()
        ax.set_xticks(range(len(nb))); ax.set_xticklabels(nb, rotation=80, style="italic")
        prev = None
        for j, w in enumerate(nb):
            if fam[w] != prev:
                k_ = sum(1 for x in nb if fam[x] == fam[w]); ax.text(j - 0.4 + (k_ - 1) / 2 + 0.4, -1.2, fam[w][:4] + ".", ha="center", fontsize=5, color="#555555", rotation=45)
                prev = fam[w]
        ax.set_xlim(-0.6, len(nb) - 0.4)
        ax.text(len(nb) - 0.3, kb / 2, "Bacteria", rotation=-90, va="center", fontsize=7, fontweight="bold")
        ax.text(len(nb) - 0.3, kb + (len(asv) - kb) / 2, "Fungi", rotation=-90, va="center", fontsize=7, fontweight="bold")
        h = [Line2D([], [], marker="o", ls="", color=c, label=s, markersize=4) for s, c in SOILCOL.items()]
        ax.legend(handles=h, frameon=False, loc="upper left", bbox_to_anchor=(1.03, 1.0))
        return _save(fig, out), d


# ---------------- figures carried over (Figs S8, S14, S16; soil texture)
SYN = {'Centaurea cyanus': 'Cyanus segetum', 'Bromus sterilis': 'Anisantha sterilis', 'Festuca arundinacea': 'Lolium arundinaceum'}

def _family_order(meta):
    f = meta[~meta.species.isin(CTRL)].groupby("species").family.first()
    return f.sort_index().sort_values(kind="stable")



def fig_glomeromycota(DF, out):
    otu = pd.read_csv(os.path.join(DF, "otu_fungi_table_R_647ASVs.csv"), sep="\t", index_col=0).T.astype(float)
    tax = pd.read_csv(os.path.join(DF, "tax_fungi_R_647ASVs.csv"), sep=None, engine="python", index_col=0)
    meta = pd.read_csv(os.path.join(DF, "metadata_fungi_647ASVs.csv")).set_index("sample")
    meta = meta.loc[meta.index.intersection(otu.index)].copy()
    glom = [c for c in otu.columns if str(tax.Phylum.get(c, "")).startswith("Glomeromycota")]
    X = otu.loc[meta.index]
    meta["glom"] = 100 * X[glom].sum(1) / X.sum(1)
    meta["soilL"] = meta.soil.map(SOILCODE)
    meta["fam"] = np.where(meta.species.isin(CTRL), "Controls (wheat)", meta.family)
    fams = sorted(meta.loc[~meta.species.isin(CTRL), "fam"].unique()) + ["Controls (wheat)"]
    with plt.rc_context(RC):
        fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw=dict(width_ratios=[1, 5.2], wspace=0.12), sharey=True)
        rng = np.random.default_rng(1)
        for i, s in enumerate(SOILCOL):
            v = meta.loc[meta.soilL == s, "glom"]
            a.boxplot(v, positions=[i], widths=0.55, showfliers=False, medianprops=dict(color="black"),
                      boxprops=dict(color=SOILCOL[s]), whiskerprops=dict(color=SOILCOL[s]), capprops=dict(color=SOILCOL[s]))
            a.scatter(i + rng.uniform(-0.18, 0.18, len(v)), v, s=3, color=SOILCOL[s], alpha=0.5, lw=0)
        a.set_xticks([0, 1]); a.set_xticklabels(["Cereal-\ndominated", "Mixed-\nsystem"])
        a.set_ylabel("Glomeromycota (% of fungal reads)"); a.set_title("Soil", loc="left")
        for j, f in enumerate(fams):
            for k, s in enumerate(SOILCOL):
                v = meta.loc[(meta.fam == f) & (meta.soilL == s), "glom"]
                if len(v) == 0: continue
                x = j + (k - 0.5) * 0.36
                b.boxplot(v, positions=[x], widths=0.3, showfliers=False, medianprops=dict(color="black", lw=0.8),
                          boxprops=dict(color=SOILCOL[s], lw=0.7), whiskerprops=dict(color=SOILCOL[s], lw=0.7), capprops=dict(color=SOILCOL[s], lw=0.7))
                b.scatter(x + rng.uniform(-0.08, 0.08, len(v)), v, s=2, color=SOILCOL[s], alpha=0.5, lw=0)
        b.set_xticks(range(len(fams))); b.set_xticklabels(fams, rotation=45, ha="right")
        b.set_title("Neighbour family", loc="left")
        for s, c in SOILCOL.items(): b.plot([], [], color=c, lw=4, label=s)
        b.legend(frameon=False, loc="upper right")
        a.margins(y=0.03)
        return _save(fig, out), meta.groupby("soilL").glom.agg(["mean", "median", "size"]).round(2)



def fig_trait_F(trait_comp_csv, out):
    comp = pd.read_csv(trait_comp_csv)
    lab = {"Distance": "Phylogenetic distance to wheat", "Segetal": "Segetal status", "Nitrogen": "Ellenberg N", "Light": "Ellenberg L",
           "Btot": "Neighbour biomass (Btot)", "SRL": "Specific root length", "RDMC": "Root dry matter content", "RootShoot": "Root:shoot ratio"}
    traits = list(lab)[::-1]
    cells = [("bacteria", "Cereal-dominated (AF193)"), ("bacteria", "Mixed-system (AF190)"), ("fungi", "Cereal-dominated (AF193)"), ("fungi", "Mixed-system (AF190)")]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.6), sharey=True, sharex=True, gridspec_kw=dict(wspace=0.06))
        for ax, (kg, s) in zip(axes, cells):
            g = comp[(comp.kingdom == kg) & (comp.soil == s)].set_index("trait")
            for y, t in enumerate(traits):
                r = g.loc[t]; P = r.P_pot if t == "Btot" else r.P_species
                if P < 0.05:   mk = dict(marker="o", s=26, color=SOILCOL[s], edgecolor="black", lw=0.5)
                elif r.P_pot < 0.05: mk = dict(marker="o", s=26, facecolor="white", edgecolor=SOILCOL[s], lw=1.0)
                else:          mk = dict(marker="o", s=10, color="#bbbbbb", lw=0)
                ax.scatter(r.F, y, zorder=3, **mk)
                ax.plot([1, r.F], [y, y], color="#cccccc", lw=0.6, zorder=1)
            ax.axvline(1, color="black", lw=0.5, ls=":")
            ax.set_title(f"{kg.capitalize()}\n{s}", loc="left"); ax.set_xlabel("db-RDA pseudo-F")
        axes[0].set_yticks(range(len(traits))); axes[0].set_yticklabels([lab[t] for t in traits])
        axes[3].scatter([], [], marker="o", s=26, color="#777777", edgecolor="black", lw=0.5, label="P < 0.05, species-level permutation")
        axes[3].scatter([], [], marker="o", s=26, facecolor="white", edgecolor="#777777", lw=1.0, label="P < 0.05 only if pots permuted")
        axes[3].scatter([], [], marker="o", s=10, color="#bbbbbb", label="P ≥ 0.05")
        fig.legend(frameon=False, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.12))
        for ax in axes: ax.margins(x=0.08)
        return _save(fig, out)



def fig_meqo_performance(s7_csv, out):
    t = pd.read_csv(s7_csv)
    cols = [("R2_insample", "In-\nsample", "#8B2E2E"), ("null_R2_mean", "Shuffled-\nbiomass\nsearch", "#9aa3ad"),
            ("R2_cv", "Cross-\nvalidated", "#4a78b8"), ("R2_oos", "Out-of-\nsample", "#3e6b3e")]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(2, 2, figsize=(5.6, 4.6), sharey=True, gridspec_kw=dict(wspace=0.08, hspace=0.55))
        axes = axes.ravel()
        for ax, (_, r) in zip(axes, t.iterrows()):
            for i, (c, l, col) in enumerate(cols):
                v = max(r[c], -0.29)
                ax.bar(i, v, color=col, width=0.7)
                if r[c] < -0.29:
                    ax.text(i, -0.28, f"{r[c]:.2f}", ha="center", va="bottom", fontsize=6, color="white")
                else:
                    ax.text(i, v + (0.012 if v >= 0 else -0.012), f"{r[c]:.2f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=6)
            ax.plot([0.62, 1.38], [r.null_R2_p95] * 2, color="black", lw=0.8)
            ax.text(1.0, r.null_R2_p95 + 0.01, "95th pct", fontsize=5.5, ha="center", va="bottom")
            ax.axhline(0, color="black", lw=0.5)
            ax.set_xticks(range(4)); ax.set_xticklabels([l for _, l, _ in cols], fontsize=5.5)
            ax.set_title(f"{r.kingdom.capitalize()}, {r.soil}\nP = {r.P_null:.3f} vs shuffled biomass ({int(r.n_perm)} runs)", loc="left")
        for ax in axes[::2]: ax.set_ylabel("Wheat biomass variance\nexplained (R²)")
        axes[0].set_ylim(-0.3, 0.36)
        return _save(fig, out)




def fig_assembly_bacteria(NST_DIR, DF, out):
    """Fig. S7: bacterial assembly metrics per neighbour treatment and soil: (a) mean betaNTI, (b) taxonomic and phylogenetic NST.
    Fungal values are not shown: the ITS1 tree cannot be aligned reliably across fungal phyla (Methods S2)."""
    d = nst_table(NST_DIR); d = d[d.kingdom == "bacteria"]
    b = pd.read_csv(os.path.join(NST_DIR, "bnti_bacteria.csv"))
    b["Species"] = b.Species.str.replace("_", " ").str.replace("bursa pastoris", "bursa-pastoris").str.replace("speculum veneris", "speculum-veneris").str.replace("foenum graecum", "foenum-graecum")
    b["Species"] = b.Species.replace(NST_SYN); b["soil"] = b.Soil.map(SOILCODE)
    sp, fam = species_order(_meta(DF)); order = sp + CTRL
    lab = [s if s not in CTRL else ("wheat × wheat" if s == "wheat x wheat" else "wheat alone") for s in order]
    vals = d.merge(b[["Species", "soil", "mean_bNTI"]], on=["Species", "soil"], how="outer")
    vals = vals[vals.Species.isin(order)]; vals.to_csv(out + "_values.csv", index=False)
    with plt.rc_context(RC):
        fig, axes = plt.subplots(1, 3, figsize=(7.2, 4.8), sharey=True, gridspec_kw=dict(wspace=0.14))
        for ax, (col, ttl, xl) in zip(axes, (("mean_bNTI", "(a) Mean βNTI", "βNTI"), ("NST", "(b) Taxonomic NST", "NST"), ("pNST", "(c) Phylogenetic NST", "NST"))):
            for k, s in enumerate(SOILCOL):
                gs = vals[vals.soil == s].set_index("Species")[col].reindex(order)
                ax.scatter(gs.values, np.arange(len(order)) + (k - 0.5) * 0.3, s=9, color=SOILCOL[s], lw=0, zorder=3)
            if col == "mean_bNTI":
                for x in (-2, 2): ax.axvline(x, color="black", lw=0.6, ls="--")
                ax.set_xlim(-3, 4)
            else:
                ax.axvline(0.5, color="black", lw=0.6, ls="--"); ax.set_xlim(-0.02, 1.02); ax.set_xticks([0, 0.5, 1])
            ax.set_title(ttl, loc="left"); ax.set_xlabel(xl)
            for j in range(1, len(order)):
                if fam.get(order[j], "Wheat") != fam.get(order[j - 1], "Wheat"): ax.axhline(j - 0.5, color="#e5e5e5", lw=0.5, zorder=0)
        axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(lab, style="italic"); axes[0].invert_yaxis()
        fig.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=s, markersize=4) for s, c in SOILCOL.items()],
                   frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.05))
        return _save(fig, out)

### `main_figures` (`mf`)
Main-text figures: Fig. 1, Fig. 2, Fig. 3, Fig. 4. Used in 3.1.1, 3.1.2, 3.2.1, 3.2.2.

In [ ]:
%%module main_figures mf
# -*- coding: utf-8 -*-
"""Main-text figures (Times New Roman). Called from Paper_figures.ipynb.
  fig1(DF, out)                    PCoA, variance partition (sequential PERMANOVA), phylum composition
  fig2(core_csv, DF, out)          ASV-level FDR core 
  fig3(DF, out)                    wheat biomass by neighbour and soil
  fig4(cf, assoc, module_dir, res_dir, out_dir)   convergence_figures.fig4_ranking_and_neighbour in Times
All functions write PNG (400 dpi), PDF and SVG (text kept as text). Soil colours as everywhere in the paper."""
import os
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch
from matplotlib.lines import Line2D
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu, chi2

SOILCODE = {"resistant": "Cereal-dominated (AF193)", "Permissive": "Mixed-system (AF190)"}
SOILCOL = {"Cereal-dominated (AF193)": "#3B75AF", "Mixed-system (AF190)": "#E08A2C"}
KCOL = {"bacteria": "#3B75AF", "fungi": "#E08A2C"}
CTRL = ["wheat x wheat", "wheat alone"]
from si_figures import FAMCOL as _F21          # one family palette for Fig. 1 and Figs S8, S12
FAMCOL = dict(_F21)
TNR = {"font.family": "serif", "font.serif": ["Times New Roman", "Times"], "mathtext.fontset": "custom",
       "mathtext.rm": "Times New Roman", "mathtext.it": "Times New Roman:italic", "font.size": 8, "axes.labelsize": 8,
       "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.spines.top": False,
       "axes.spines.right": False, "axes.linewidth": 0.6, "pdf.fonttype": 42, "svg.fonttype": "none"}


def _save(fig, out):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{out}.{ext}", dpi=400, bbox_inches="tight")
    plt.close(fig)
    return out + ".png"


def _load(DF, kg):
    n = 1314 if kg == "bacteria" else 647
    X = pd.read_csv(os.path.join(DF, f"otu_{kg}_table_R_{n}ASVs.csv"), sep="\t", index_col=0).T.astype(float)
    md = pd.read_csv(os.path.join(DF, f"metadata_{kg}_{n}ASVs.csv")).set_index("sample")
    md = md.loc[md.index.intersection(X.index)]
    tax = pd.read_csv(os.path.join(DF, f"tax_{kg}_R_{n}ASVs.csv"), sep=None, engine="python", index_col=0)
    return X.loc[md.index], md, tax


def pcoa(D):
    n = D.shape[0]; J = np.eye(n) - 1 / n; B = -0.5 * J @ (D ** 2) @ J
    w, V = np.linalg.eigh(B); o = np.argsort(w)[::-1]; w, V = w[o], V[:, o]
    pos = w > 0
    return V[:, :2] * np.sqrt(w[:2]), 100 * w[:2] / w[pos].sum()


def _ellipse(ax, xy, col, q=0.95):
    mu = xy.mean(0); C = np.cov(xy.T); vals, vecs = np.linalg.eigh(C)
    k = np.sqrt(chi2.ppf(q, 2)); ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse(mu, 2 * k * np.sqrt(vals[1]), 2 * k * np.sqrt(vals[0]), angle=ang, fc=col, ec=col, alpha=0.10, lw=0))
    ax.add_patch(Ellipse(mu, 2 * k * np.sqrt(vals[1]), 2 * k * np.sqrt(vals[0]), angle=ang, fc="none", ec=col, lw=1.0))


def table1_from_docx(path):
    """Read Table 1 (sequential PERMANOVA) from the manuscript: {kingdom: {soil|species|inter: (R2 %, F)}}."""
    from docx import Document
    tb = next(t for t in Document(path).tables if t.rows[0].cells[0].text.strip() == "Model")
    out, kg = {}, None
    key = {"Soil": "soil", "Species": "species", "Soil × Species": "inter"}
    for r in tb.rows[1:]:
        c = [x.text.strip() for x in r.cells]
        if c[0]: kg = "bacteria" if c[0].startswith("Bacterial") else "fungi"
        if c[1] in key: out.setdefault(kg, {})[key[c[1]]] = (float(c[3]), float(c[4]))
    return out


def table1_from_csv(path):
    """Read Table 1 as written by Paper_analyses.qmd (Table1_variance_partition.csv): {kingdom: {soil|species|inter: (R2 %, F)}}."""
    t = pd.read_csv(path)
    def key(term):                      
        term = str(term)
        if term.startswith("Soil of origin"): return "soil"
        if term.startswith("Neighbour identity"): return "species"
        if term.startswith("Soil") and "neighbour" in term: return "inter"
        return None
    out = {}
    for _, r in t.iterrows():
        k = key(r.Term)
        if k is None: continue
        kg = "bacteria" if r.Kingdom.startswith("Bacteria") else "fungi"
        out.setdefault(kg, {})[k] = (float(r.R2_pct), float(r.F))
    return out


def fig1(DF, out, table1):
    """table1: {kingdom: {'soil': (R2%, F), 'species': (R2%, F), 'inter': (R2%, F)}}, e.g. table1_from_docx(manuscript)."""
    with plt.rc_context(TNR):
        fig = plt.figure(figsize=(7.2, 6.4))
        gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.95], height_ratios=[1.15, 1], wspace=0.34, hspace=0.42)
        famlab = {}
        for j, kg in enumerate(("bacteria", "fungi")):
            X, md, _ = _load(DF, kg)
            D = squareform(pdist(X.values, "braycurtis")); Y, ev = pcoa(D)
            ax = fig.add_subplot(gs[0, j]); md = md.assign(PC1=Y[:, 0], PC2=Y[:, 1], soilL=md.soil.map(SOILCODE))
            for s, c in SOILCOL.items():
                g = md[md.soilL == s]; ax.scatter(g.PC1, g.PC2, s=2.5, color=c, alpha=0.35, lw=0, zorder=1)
                _ellipse(ax, g[["PC1", "PC2"]].values, c)
            md["fam"] = np.where(md.species.isin(CTRL), "Wheat", md.family)
            cen = md.groupby(["soilL", "species", "fam"])[["PC1", "PC2"]].mean().reset_index()
            for (s, mk) in zip(SOILCOL, ("o", "^")):
                g = cen[cen.soilL == s]
                ax.scatter(g.PC1, g.PC2, s=16, marker=mk, c=[FAMCOL[f] for f in g.fam], edgecolor="black", lw=0.3, zorder=3)
            famlab.update({f: FAMCOL[f] for f in cen.fam.unique()})
            ax.set_xlabel(f"PCoA axis 1 ({ev[0]:.1f}%)"); ax.set_ylabel(f"PCoA axis 2 ({ev[1]:.1f}%)")
            ax.set_title(("Bacteria (rpoB)" if kg == "bacteria" else "Fungi (ITS1)"), loc="left", fontweight="bold")
            t = table1[kg]
            ax.text(0.02, 0.99, f"Soil R² = {t['soil'][0]:.1f}%\nSpecies R² = {t['species'][0]:.1f}%\nSoil × species R² = {t['inter'][0]:.1f}%",
                    transform=ax.transAxes, fontsize=6, va="top", style="italic", linespacing=1.2)
            ax.margins(y=0.12)
        axl = fig.add_subplot(gs[0, 2]); axl.axis("off")
        h = [Patch(fc=c, ec=c, alpha=0.4, label=s.replace(" (", "\n(")) for s, c in SOILCOL.items()]
        h += [Line2D([], [], ls="", marker="o", mfc="white", mec="black", label="centroid, cereal-dominated"),
              Line2D([], [], ls="", marker="^", mfc="white", mec="black", label="centroid, mixed-system")]
        h += [Line2D([], [], ls="", marker="s", mfc=famlab[f], mec="black", mew=0.3, label=f) for f in sorted(famlab, key=lambda x: (x == "Wheat", x))]
        h[-1].set_label("Wheat (controls)")
        axl.legend(handles=h, frameon=False, loc="upper left", fontsize=6.5, title="Soil and neighbour family", title_fontsize=7)
        # (b) variance partition
        axb = fig.add_subplot(gs[1, 0:2])
        terms = [("soil", "Soil"), ("species", "Species"), ("inter", "Soil × species")]
        for i, (k, lab) in enumerate(terms):
            for dy, kg in ((-0.18, "bacteria"), (0.18, "fungi")):
                r2, F = table1[kg][k]
                axb.barh(i + dy, r2, height=0.34, color=KCOL[kg], lw=0)
                axb.text(r2 + 0.15, i + dy, f"{r2:.1f}%  (F = {F:.2f})", va="center", fontsize=6.5)
        axb.set_yticks(range(3)); axb.set_yticklabels([l for _, l in terms]); axb.invert_yaxis()
        axb.set_xlim(0, 14.5); axb.set_xlabel("Variance explained (R², %)")
        axb.legend(handles=[Patch(color=KCOL["bacteria"], label="Bacteria"), Patch(color=KCOL["fungi"], label="Fungi")], frameon=False, loc="upper right")
        axb.set_title("Sequential PERMANOVA (Bray–Curtis, 999 permutations)", loc="left")
        # (c, d) phylum composition, all samples pooled
        sub = gs[1, 2].subgridspec(2, 1, hspace=0.95)
        for r_, kg in enumerate(("fungi", "bacteria")):
            X, md, tax = _load(DF, kg)
            ph = tax.Phylum.reindex(X.columns).fillna("unidentified").astype(str)
            n_asv = ph.value_counts(); ra = (X.div(X.sum(1), axis=0).mean()).groupby(ph).sum()
            top = list(ra.sort_values(ascending=False).index[:7])
            n_asv = n_asv.groupby(lambda p: p if p in top else "other").sum(); ra = ra.groupby(lambda p: p if p in top else "other").sum()
            order = [p for p in ra.sort_values(ascending=False).index if p != "other"] + (["other"] if "other" in ra.index else [])
            cmap = plt.get_cmap("tab10"); colp = {p: (cmap(i) if p != "other" else "#cccccc") for i, p in enumerate(order)}
            axc = fig.add_subplot(sub[r_]); b1 = b2 = 0
            for p in order:
                v1 = n_asv.get(p, 0) / n_asv.sum(); v2 = ra.get(p, 0) / ra.sum()
                axc.bar(0, v1, bottom=b1, color=colp[p], width=0.7, lw=0); axc.bar(1, v2, bottom=b2, color=colp[p], width=0.7, lw=0, label=p)
                b1 += v1; b2 += v2
            axc.set_xticks([0, 1]); axc.set_xticklabels([f"ASVs\n(n = {n_asv.sum():,})", "Relative\nabundance"], fontsize=6.5)
            axc.set_ylim(0, 1); axc.set_yticks([0, 0.5, 1]); axc.set_ylabel("Proportion"); axc.set_title(("Fungi" if kg == "fungi" else "Bacteria"), loc="left", fontweight="bold", pad=3)
            axc.text(-0.34, 1.10, "(c)" if kg == "fungi" else "(d)", transform=axc.transAxes, fontsize=10, fontweight="bold", va="bottom")
            axc.legend(frameon=False, fontsize=5.8, loc="center left", bbox_to_anchor=(1.0, 0.5), handlelength=1)
        fig.axes[0].text(-0.28, 1.06, "(a)", transform=fig.axes[0].transAxes, fontsize=10, fontweight="bold", va="bottom")
        axb.text(-0.14, 1.06, "(b)", transform=axb.transAxes, fontsize=10, fontweight="bold", va="bottom")
        return _save(fig, out)


def fig2(P, lfc, T, fam, out):
    """ASV x neighbour calls passing >= 2 of 3 methods at FDR-adjusted q < 0.05 (log2FC > 2).
    P: {(kingdom, soil): set of (neighbour, ASV)}; soil keys 'Resistant'/'Permissive' as in da_consensus;
    lfc: {(kingdom, soil, neighbour, ASV): DESeq2 log2FC}; T[kingdom]['tax']: taxonomy; fam: {neighbour: family}."""
    rows = [dict(kingdom=kg, soil=SOILCODE["resistant" if s == "Resistant" else "Permissive"], neighbour=w, ASV=c,
                 l=lfc.get((kg, s, w, c), np.nan)) for (kg, s), pp in P.items() for w, c in pp]
    d = pd.DataFrame(rows)
    d["genus"] = [T[k]["tax"].Genus.get(a) for k, a in zip(d.kingdom, d.ASV)]
    d["phylum"] = [T[k]["tax"].Phylum.get(a) for k, a in zip(d.kingdom, d.ASV)]
    d.sort_values(["kingdom", "soil", "neighbour", "ASV"]).to_csv(out + "_values.csv", index=False)
    nb = sorted(fam, key=lambda s: (fam[s], s))
    asv = d.groupby(["kingdom", "phylum", "genus", "ASV"]).neighbour.nunique().reset_index() \
           .sort_values(["kingdom", "phylum", "genus", "neighbour"], ascending=[True, True, True, False])
    lab = [f"{g} {a.replace('Cluster_', 'ASV_')}" for g, a in zip(asv.genus, asv.ASV)]
    ypos = {(k, a): i for i, (k, a) in enumerate(zip(asv.kingdom, asv.ASV))}
    with plt.rc_context(TNR):
        fig, ax = plt.subplots(figsize=(7.4, 0.205 * len(asv) + 2.6))
        for k, s in enumerate(SOILCOL):
            g = d[d.soil == s]
            ax.scatter([nb.index(w) for w in g.neighbour], [ypos[(a, b)] for a, b in zip(g.kingdom, g.ASV)],
                       s=30, color=SOILCOL[s], lw=0.3, edgecolor="white", label=s, zorder=3)
        for i in range(len(asv)):
            ax.axhline(i, color="#f0f0f0", lw=0.4, zorder=0)
        bounds, prev = [], None
        for j, w in enumerate(nb):
            if fam[w] != prev: bounds.append(j); prev = fam[w]
        bounds.append(len(nb))
        for j in bounds[1:-1]: ax.axvline(j - 0.5, color="#bbbbbb", lw=0.6, zorder=1)
        kb = (asv.kingdom == "bacteria").sum(); ax.axhline(kb - 0.5, color="black", lw=0.8)
        ax.set_yticks(range(len(asv))); ax.set_yticklabels(lab, style="italic", fontsize=8); ax.invert_yaxis()
        ax.set_xticks(range(len(nb))); ax.set_xticklabels(nb, rotation=75, style="italic", fontsize=8, ha="right", rotation_mode="anchor")
        ax.set_xlim(-0.6, len(nb) - 0.4); ax.set_ylim(len(asv) - 0.4, -0.6)
        for a0, a1 in zip(bounds[:-1], bounds[1:]):
            ax.text((a0 + a1 - 1) / 2, -0.9, fam[nb[a0]], ha="left", va="bottom", rotation=50, fontsize=8, rotation_mode="anchor")
        ax.text(len(nb) - 0.1, (kb - 1) / 2, "Bacteria", rotation=-90, va="center", fontsize=9, fontweight="bold")
        ax.text(len(nb) - 0.1, kb + (len(asv) - kb - 1) / 2, "Fungi", rotation=-90, va="center", fontsize=9, fontweight="bold")
        ax.spines["top"].set_visible(True); ax.spines["right"].set_visible(True)
        h = [Line2D([], [], marker="o", ls="", color=c, label=s, markersize=6) for s, c in SOILCOL.items()]
        ax.legend(handles=h, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.215), ncol=2, fontsize=8.5)
        return _save(fig, out)


def fig3(DF, out, star=None):
    """Wheat biomass by neighbour and soil. star: treatments whose wheat biomass differs between soils (two-sided
    Mann–Whitney, Benjamini–Hochberg q < 0.05 across the 30 treatments); computed here when None."""
    md = pd.read_csv(os.path.join(DF, "metadata_bacteria_1314ASVs.csv")).dropna(subset=["Dry_wheat_mass"])
    md["soilL"] = md.soil.map(SOILCODE)
    fam = md[~md.species.isin(CTRL)].groupby("species").family.first()
    sp = list(fam.reset_index().sort_values(["family", "species"]).species)
    order = sp + ["wheat alone", "wheat x wheat"]
    if star is None:   # two-sided Mann-Whitney between soils, Benjamini-Hochberg across the 30 treatments, q < 0.05
        from statsmodels.stats.multitest import multipletests
        pv = [mannwhitneyu(*[md[(md.species == s) & (md.soilL == L)].Dry_wheat_mass for L in SOILCOL], alternative="two-sided").pvalue for s in order]
        q = multipletests(pv, method="fdr_bh")[1]
        star = [s for s, qq in zip(order, q) if qq < 0.05]
    ctrl = md[md.species.isin(CTRL)].groupby("species").Dry_wheat_mass.mean()
    with plt.rc_context(TNR):
        fig, ax = plt.subplots(figsize=(7.2, 3.6))
        for i, x in enumerate(order):
            for k, s in enumerate(SOILCOL):
                v = md[(md.species == x) & (md.soilL == s)].Dry_wheat_mass.values
                bp = ax.boxplot(v, positions=[i + (k - 0.5) * 0.38], widths=0.34, patch_artist=True, showfliers=True,
                                flierprops=dict(marker="o", markersize=1.8, markerfacecolor="black", markeredgecolor="none"),
                                medianprops=dict(color="black", lw=0.7), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
                bp["boxes"][0].set(facecolor=SOILCOL[s], edgecolor="black", lw=0.4)
            if x in star:
                top = md[md.species == x].Dry_wheat_mass.max(); ax.text(i, top + 0.04, "*", ha="center", va="bottom", fontsize=11, color="#c00000")
        ax.axhline(ctrl["wheat x wheat"], color="#1f3fbf", lw=0.8, ls="--"); ax.axhline(ctrl["wheat alone"], color="#c00000", lw=0.8, ls="--")
        famseq = [fam.get(x, "Control") for x in order]
        for j in range(1, len(order)):
            if famseq[j] != famseq[j - 1]: ax.axvline(j - 0.5, color="#aaaaaa", lw=0.5, ls="--", zorder=0)
        ymax = md.Dry_wheat_mass.max() * 1.12
        for f in ("Asteraceae", "Fabaceae", "Poaceae"):
            idx = [j for j, g in enumerate(famseq) if g == f]
            ax.text(np.mean(idx), ymax * 0.97, f, ha="center", va="top", fontsize=8.5)
        ax.set_ylim(0, ymax); ax.set_xlim(-0.6, len(order) - 0.4)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels([("wheat × wheat" if o == "wheat x wheat" else o) for o in order], rotation=80, style="italic", fontsize=8)
        ax.set_ylabel("Wheat aboveground dry mass (g)")
        h = [Patch(fc=c, ec="black", lw=0.4, label=s) for s, c in SOILCOL.items()] + \
            [Line2D([], [], color="#c00000", ls="--", lw=0.8, label="mean, wheat alone"), Line2D([], [], color="#1f3fbf", ls="--", lw=0.8, label="mean, wheat × wheat")]
        ax.legend(handles=h, frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.50), fontsize=8)
        return _save(fig, out), star


def fig4(scores_csv, tests_csv, out, label_ends=3):
    """Fig. 4: species-centroid db-RDA of endosphere composition on species-mean wheat biomass (28 weeds per panel)"""
    sc = pd.read_csv(scores_csv); tt = pd.read_csv(tests_csv).set_index(["kingdom", "soil"])
    GRP = {"Poaceae": ("Poaceae (monocot)", "#1B7837"), "Juncaceae": ("Juncaceae (monocot)", "#A6DBA0"),
           "Fabaceae": ("Fabaceae (legume)", "#9C3D8F")}
    OTHER = ("other eudicots", "#E6E6E6")
    abbr = lambda s: s.split()[0][0] + ". " + " ".join(s.split()[1:])
    with plt.rc_context(TNR):
        fig, axs = plt.subplots(2, 2, figsize=(7.0, 6.4))
        for k, (ax, (kg, so)) in enumerate(zip(axs.flat, [(kg, so) for kg in ("bacteria", "fungi") for so in SOILCOL])):
            d = sc[(sc.kingdom == kg) & (sc.soil == so)].copy(); r = tt.loc[(kg, so)]
            if np.corrcoef(d.CAP1, d.biomass)[0, 1] < 0: d["CAP1"] = -d.CAP1          # biomass increases to the right
            b = d.biomass; size = 15 + 160 * (b - b.min()) / (b.max() - b.min())
            c = [GRP.get(f, OTHER)[1] for f in d.family]
            order = np.argsort(-size.values)                                            # large points first, small on top
            ax.scatter(d.CAP1.values[order], d.MDS1.values[order], s=size.values[order], c=[c[i] for i in order],
                       edgecolor="black", lw=0.4, zorder=3)
            ax.axhline(0, color="#dddddd", lw=0.5, zorder=0); ax.axvline(0, color="#dddddd", lw=0.5, zorder=0)
            if label_ends:
                ds = d.sort_values("CAP1"); yr = d.MDS1.max() - d.MDS1.min()
                for end, sub, ha, dx in (("low", ds.head(label_ends), "right", -5), ("high", ds.tail(label_ends), "left", 5)):
                    sub = sub.sort_values("MDS1"); last = None; dy = 0
                    for _, row in sub.iterrows():
                        dy = (dy + 7 if (last is not None and row.MDS1 - last < 0.07 * yr) else 0); last = row.MDS1
                        rad = np.sqrt(15 + 160 * (row.biomass - b.min()) / (b.max() - b.min())) / 2     # marker radius, points
                        ax.annotate(abbr(row.species), (row.CAP1, row.MDS1), xytext=(np.sign(dx) * (rad + 2), dy), textcoords="offset points",
                                    ha=ha, va="center", fontsize=6, style="italic", color="#333333", zorder=4)
            xr = d.CAP1.max() - d.CAP1.min(); ax.set_xlim(d.CAP1.min() - 0.42 * xr, d.CAP1.max() + 0.40 * xr); ax.margins(y=0.10)
            ax.set_xlabel(f"db-RDA1, wheat biomass ({d.eig_CAP1_pct.iloc[0]:.1f}%)")
            ax.set_ylabel(f"MDS1, unconstrained ({d.eig_MDS1_pct.iloc[0]:.1f}%)")
            ttl = f"{'Bacteria' if kg == 'bacteria' else 'Fungi'}, {'cereal-dominated' if so.startswith('Cereal') else 'mixed-system'} soil"
            pstr = f"P = {r.P:.3f}" if r.P >= 0.001 else "P < 0.001"
            ax.set_title(f"({'abcd'[k]}) {ttl}\nR² = {r.R2_pct:.1f}%, {pstr}, q = {r.q_BH:.2f}", loc="left")
        h = [Line2D([], [], ls="", marker="o", mfc=v[1], mec="black", mew=0.4, markersize=6, label=v[0]) for v in list(GRP.values()) + [OTHER]]
        fig.legend(handles=h, loc="lower center", ncol=4, frameon=False,
                   title="Neighbour group; point size, species-mean wheat biomass", bbox_to_anchor=(0.5, -0.01))
        fig.tight_layout(rect=(0, 0.06, 1, 1))
        return _save(fig, out)

### `da_consensus` (`dc`)
Differential-abundance helpers: reading the DESeq2 / edgeR / metagenomeSeq exports, orientation check, thresholds. Used in 3.1.2.

In [ ]:
%%module da_consensus dc
# -*- coding: utf-8 -*-
"""
High-confidence neighbour-enriched taxa (Fig. 2, Table S3, Notes S3, Results 3.1.2).

A taxon is retained for a (soil, neighbour) pair when at least two of DESeq2, edgeR and
metagenomeSeq call it ENRICHED under the neighbour relative to wheat x wheat:
    DESeq2         p < 0.05      and log2FC(neighbour/control) >  2
    edgeR          PValue < 0.01 and logFC(neighbour/control)  >  2
    metagenomeSeq  adj. p < 0.05 and coefficient(neighbour)    >  0
These are the thresholds of the original analysis (Analyses_bacteria_v1.2.ipynb, cell 77).

"""
import glob, os, re
import numpy as np
import pandas as pd

CONTROL = "wheat x wheat"
SYN = {"Centaurea cyanus": "Cyanus segetum", "Bromus sterilis": "Anisantha sterilis",
       "Festuca arundinacea": "Lolium arundinaceum",
       "Festuca myuros": "Lolium arundinaceum",   # mislabelled in the DESeq2 / metagenomeSeq exports
       "Viola arvensis": "Vulpia myuros"}        
SOILCODE = {"Resistant": "resistant", "Permissive": "Permissive"}
SOILLABEL = {"Resistant": "cereal-dominated", "Permissive": "mixed-system"}
THR = {"DESeq2": ("p", 0.05, 2.0), "edgeR": ("p", 0.01, 2.0), "metagenomeSeq": ("p", 0.05, 0.0)}
FILES = {"DESeq2": ("DESeq2_species", "DESeq_{kg}_{soil}_species_vs_wheat_x_wheat_pval.csv"),
         "edgeR": ("edgeR_species", "edgeR_{kg}_{soil}_vs_wheat_x_wheat.csv"),
         "metagenomeSeq": ("metagenomeSeq_species", "MetaSeq_{kg}_{soil}_CSS_wheat_x_wheat.csv")}


def norm_name(s):
    s = str(s).replace(".", " ").strip()
    for a, b in (("bursa pastoris", "bursa-pastoris"), ("speculum veneris", "speculum-veneris"),
                 ("foenum graecum", "foenum-graecum")):
        s = s.replace(a, b)
    return SYN.get(s, s)


def load_tables(matrix_dir):
    T = {}
    for kg, tag in (("bacteria", "1314"), ("fungi", "647")):
        otu = pd.read_csv(os.path.join(matrix_dir, f"otu_{kg}_table_R_{tag}ASVs.csv"), sep="\t", index_col=0).T.astype(float)
        md = pd.read_csv(os.path.join(matrix_dir, f"metadata_{kg}_{tag}ASVs.csv")).set_index("sample")
        md["species"] = md["species"].map(norm_name)
        otu = otu.loc[otu.index.intersection(md.index)]
        tax = pd.read_csv(os.path.join(matrix_dir, f"tax_{kg}_R_{tag}ASVs.csv"), sep="\t", index_col=0)
        T[kg] = dict(X=otu.div(otu.sum(1), axis=0), md=md, tax=tax)
    return T


def raw_lfc(T, kg, soil, weed):
    X, md = T[kg]["X"], T[kg]["md"]
    s = md[md.soil == SOILCODE[soil]]
    ia = s.index[s.species == weed].intersection(X.index)
    ic = s.index[s.species == CONTROL].intersection(X.index)
    if len(ia) == 0 or len(ic) == 0:
        return None
    return np.log2((X.loc[ia].mean() * 7000 + 0.5) / (X.loc[ic].mean() * 7000 + 0.5))


def read_export(path, method):
    """Long table: weed, Cluster, lfc (as exported), p."""
    d = pd.read_csv(path)
    if method == "metagenomeSeq":
        key = d.iloc[:, 0].astype(str)
        d, key = d[key.str.contains(".Cluster_", regex=False)], key[key.str.contains(".Cluster_", regex=False)]  # drop ".NA" rows
        return pd.DataFrame({"weed": key.str.split(".Cluster_").str[0].map(norm_name),
                             "Cluster": "Cluster_" + key.str.split(".Cluster_").str[1],
                             "lfc": d.iloc[:, 1].astype(float), "p": d["adjPvalues"].astype(float)})
    a = d["comparison"].str.split(" vs ", expand=True)
    a0, a1 = a[0].map(norm_name), a[1].map(norm_name)
    if not ((a0 == CONTROL) | (a1 == CONTROL)).all():
        raise ValueError(f"{os.path.basename(path)}: rows without the {CONTROL} contrast")
    lfc = d["log2FoldChange" if method == "DESeq2" else "logFC"].astype(float)
    # express everything as if the control were named first, so one sign applies per file
    lfc = np.where(a0 == CONTROL, lfc, -lfc)
    return pd.DataFrame({"weed": np.where(a0 == CONTROL, a1, a0), "Cluster": d["taxa"].astype(str), "lfc": lfc,
                         "p": d["pvalue" if method == "DESeq2" else "PValue"].astype(float)})


def orient_and_check(T, kg, soil, df, fname, min_r=0.5, label_margin=0.1):
    """Sign that turns df.lfc into log2(neighbour/control); verifies every neighbour label."""
    weeds = sorted(set(df.weed.unique()) - {"wheat alone"}); raws = {w: raw_lfc(T, kg, soil, w) for w in T[kg]["md"].species.unique()
                                             if w != CONTROL and w != "wheat alone"}
    rs, audit = [], []
    for w in weeds:
        g = df[df.weed == w]
        if w not in raws or raws[w] is None:
            raise ValueError(f"{fname}: neighbour '{w}' not in the metadata")
        cor = {}
        for cand, r in raws.items():
            if r is None: continue
            t = r.reindex(g.Cluster.values).values; ok = ~np.isnan(t)
            if ok.sum() >= 10: cor[cand] = np.corrcoef(g.lfc.values[ok], t[ok])[0, 1]
        if w not in cor: continue
        best = max(cor, key=lambda c: abs(cor[c]))
        # A label is wrong when another neighbour fits clearly better. Small exports (a few
        # dozen rows, mostly taxa absent from the control) correlate with every neighbour,
        # so near-ties are not evidence of a mix-up; the known mix-up differs by ~0.3.
        ok_ = abs(cor[w]) >= abs(cor[best]) - label_margin
        audit.append(dict(file=fname, neighbour=w, n_rows=len(g), r=round(cor[w], 3), best_match=best,
                          r_best=round(cor[best], 3), label_ok=ok_))
        rs.append(cor[w])
    med = float(np.median(rs))
    if abs(med) < min_r:
        raise ValueError(f"{fname}: median r with raw log2FC = {med:.2f}; file does not match the abundance table")
    bad = [a for a in audit if not a["label_ok"]]
    if bad:
        raise ValueError(f"{fname}: neighbour labels do not match the data: {bad[:3]}")
    return (1 if med > 0 else -1), med, audit


def enriched(T, roots, kg, soil):
    """All (method, neighbour, Cluster) calls enriched under the neighbour, plus a file audit."""
    calls, audit = [], []
    for m, (sub, pat) in FILES.items():
        f = next((os.path.join(r, sub, pat.format(kg=kg, soil=soil)) for r in roots
                  if os.path.exists(os.path.join(r, sub, pat.format(kg=kg, soil=soil)))), None)
        if f is None:
            raise FileNotFoundError(f"{m} export for {kg} {soil} not found under {roots}")
        df = read_export(f, m)
        df = df[df.weed != "wheat alone"]          # control-vs-control contrast, not a neighbour
        sign, med, a = orient_and_check(T, kg, soil, df, os.path.basename(f))
        audit += [dict(a, method=m, kingdom=kg, soil=soil, sign_applied=sign, median_r=round(med, 3), path=f) for a in a]
        df["lfc_n_over_c"] = sign * df.lfc
        _, pthr, lthr = THR[m]
        k = (df.p < pthr) & (df.lfc_n_over_c > lthr)
        calls.append(df.loc[k, ["weed", "Cluster"]].assign(Method=m))
    return pd.concat(calls, ignore_index=True).drop_duplicates(), pd.DataFrame(audit)


def consensus(matrix_dir, roots, min_methods=2):
    """Returns (long, audit): long has one row per method call for every retained
    (Cluster, Plant_Species, Soil) — the `filtered_df` layout the Fig. 2 builders expect."""
    T = load_tables(matrix_dir); out, aud = [], []
    for kg in ("bacteria", "fungi"):
        for soil in ("Resistant", "Permissive"):
            c, a = enriched(T, roots, kg, soil); aud.append(a)
            n = c.groupby(["Cluster", "weed"]).Method.nunique()
            keep = n[n >= min_methods].reset_index()[["Cluster", "weed"]]
            c = c.merge(keep, on=["Cluster", "weed"])
            out.append(c.rename(columns={"weed": "Plant_Species"}).assign(Kingdom=kg, Soil=soil))
    return pd.concat(out, ignore_index=True), pd.concat(aud, ignore_index=True), T


def table_s3(long, T):
    """One row per ASV, as in the published Table S3, with soil-specific neighbour lists."""
    rows = []
    for (kg, cl), g in long.groupby(["Kingdom", "Cluster"]):
        tax = T[kg]["tax"].reindex([cl]).iloc[0]
        soils = sorted({SOILLABEL[s] for s in g.Soil}, key=lambda s: s != "cereal-dominated")
        treat = "; ".join(f"{SOILLABEL[s]}: " + ", ".join(sorted(set(gg.Plant_Species)))
                          for s, gg in sorted(g.groupby("Soil"), key=lambda x: x[0] != "Resistant"))
        pair_methods = g.groupby(["Soil", "Plant_Species"]).Method.apply(lambda x: ", ".join(sorted(set(x))))
        rows.append(dict(Kingdom=kg, Cluster=cl, Phylum=tax.get("Phylum"), Genus=tax.get("Genus"),
                         Species=tax.get("Species"), Soil=", ".join(soils),
                         Methods=", ".join(sorted(set(g.Method))),
                         Method_combinations="; ".join(sorted(set(pair_methods))),
                         n_neighbours=g.groupby("Soil").Plant_Species.nunique().sum(),
                         Plant_Treatments=treat))
    t = pd.DataFrame(rows)
    t["_n"] = t.Cluster.str.extract(r"(\d+)").astype(int)
    return t.sort_values(["Kingdom", "_n"]).drop(columns="_n").reset_index(drop=True)


def summary(long):
    """Counts quoted in Results 3.1.2 / Notes S3."""
    pairs = long.drop_duplicates(["Kingdom", "Soil", "Plant_Species", "Cluster"])
    s = pairs.groupby(["Kingdom", "Soil"]).agg(ASVs=("Cluster", "nunique"), pairs=("Cluster", "size"),
                                               neighbours=("Plant_Species", "nunique")).reset_index()
    s["Soil"] = s.Soil.map(SOILLABEL)
    per = (long.groupby(["Kingdom", "Soil", "Plant_Species", "Cluster"]).Method
           .apply(lambda x: ", ".join(sorted(set(x)))).reset_index())
    combo = per.groupby(["Kingdom", "Method"]).size().unstack(fill_value=0)
    return s, combo


# ------ Fig. 2
FUNCTION_BY_GENUS = {'Rhizobium': 'Nitrogen Fixer', 'Bradyrhizobium': 'Nitrogen Fixer', 'Pseudomonas': 'PGPR + Biocontrol', 'Bacillus': 'PGPR + Biocontrol', 'Azospirillum': 'PGPR', 'Streptomyces': 'Biocontrol agent', 'Trichoderma': 'Biocontrol + Nutrient cycling', 'Burkholderia': 'Stress tolerance / PGPR', 'Paenibacillus': 'PGPR + Biocontrol', 'Enterobacter': 'PGPR', 'Azotobacter': 'Nitrogen Fixer + PGPR', 'Arthrobacter': 'Stress-tolerant', 'Serratia': 'Biocontrol', 'Frankia': 'Nitrogen Fixer', 'Mesorhizobium': 'Nitrogen Fixer', 'Sinorhizobium': 'Nitrogen Fixer', 'Azoarcus': 'Nitrogen Fixer / PGPR', 'Pantoea': 'PGPR', 'Klebsiella': 'Nitrogen Fixer', 'Herbaspirillum': 'Nitrogen Fixer / PGPR', 'Devosia': 'Rhizosphere beneficial', 'Sphingomonas': 'Stress-tolerant', 'Flavobacterium': 'PGPR', 'Acinetobacter': 'PGPR', 'Microbacterium': 'PGPR', 'Variovorax': 'Plant-associated PGPR', 'Rhodococcus': 'Biocontrol', 'Lysobacter': 'Biocontrol', 'Niastella': 'Potential PGPR', 'Polaromonas': 'Mixed (some PGPR)', 'Hyphomicrobium': 'Nitrogen cycle (unclear PGPR)', 'Agrobacterium': 'Plant-associated PGPR', 'Afipia': 'Potential PGPR', 'Nitrobacter': 'Nitrogen cycle (unclear PGPR)', 'Massilia': 'PGPR', 'Mycobacterium': 'Decomposer'}
FUNCTION_PALETTE = {'Nitrogen Fixer': '#1f77b4', 'PGPR + Biocontrol': '#ff7f0e', 'PGPR': '#2ca02c', 'Stress tolerance / PGPR': '#d62728', 'Biocontrol agent': '#9467bd', 'Biocontrol + Nutrient cycling': '#8c564b', 'Decomposer / Biocontrol': '#e377c2', 'Decomposer': '#7f7f7f', 'Potential PGPR': '#bcbd22', 'Mixed (some PGPR)': '#17becf', 'Nitrogen Fixer + PGPR': '#aec7e8', 'Rhizosphere beneficial': '#c49c94', 'Stress-tolerant': '#f7b6d2', 'Nitrogen cycle (unclear PGPR)': '#dbdb8d', 'Plant-associated PGPR': '#9edae5', 'Biocontrol': '#ff9896', 'Nitrogen Fixer / PGPR': '#98df8a', 'PGPR + Nitrogen Fixer': '#c5b0d5'}
BACT_PHYLUM_COLOURS = {'Proteobacteria': '#332288', 'Bacteroidetes': '#999933', 'Firmicutes': '#CC6677', 'Actinobacteria': '#88CCEE', 'unidentified': '#AAAAAA', 'Chloroflexi': '#882255', 'Multi-affiliation': '#000000'}


def write_table_s3(long, T, path):
    """Table S3 workbook: README, one sheet per kingdom, summary, method combinations."""
    t = table_s3(long, T); s, combo = summary(long)
    with pd.ExcelWriter(path) as xw:
        pd.DataFrame({"Table S3": [
            "High-confidence neighbour-enriched ASVs: called enriched under the neighbour relative to wheat x wheat "
            "by at least two of DESeq2 (p < 0.05, log2FC > 2), edgeR (p < 0.01, logFC > 2) and metagenomeSeq "
            "(adj. p < 0.05, coefficient > 0) for the same neighbour and soil."]}).to_excel(xw, sheet_name="README", index=False)
        for kg, sh in (("bacteria", "bacterial_clusters"), ("fungi", "fungi_clusters")):
            t[t.Kingdom == kg].drop(columns="Kingdom").to_excel(xw, sheet_name=sh, index=False)
        s.to_excel(xw, sheet_name="summary", index=False)
        combo.reset_index().to_excel(xw, sheet_name="method_combinations", index=False)
    return t


def fig2_inputs(long, T, min_combos_bacteria=5):
    """One row per (ASV, neighbour, soil)"""
    u = long.drop_duplicates(["Kingdom", "Soil", "Plant_Species", "Cluster"])
    fam = {}
    for kg in ("bacteria", "fungi"):
        fam.update(dict(zip(T[kg]["md"].species, T[kg]["md"].family)))
    fb = u[u.Kingdom == "bacteria"][["Cluster", "Plant_Species", "Soil"]].reset_index(drop=True)
    ff = u[u.Kingdom == "fungi"][["Cluster", "Plant_Species", "Soil"]].reset_index(drop=True)
    gb = T["bacteria"]["tax"].Genus.to_dict()
    combos = fb.assign(G=fb.Cluster.map(gb)).drop_duplicates(["G", "Plant_Species", "Soil"]).groupby("G").size()
    keep = set(combos[combos >= min_combos_bacteria].index)
    return dict(bact=fb[fb.Cluster.map(gb).isin(keep)], bact_all=fb, fungi=ff, family=fam,
                genus_b=gb, phylum_b=T["bacteria"]["tax"].Phylum.to_dict(),
                genus_f=T["fungi"]["tax"].Genus.to_dict(), phylum_f=T["fungi"]["tax"].Phylum.to_dict(),
                n_genera_shown=len(keep), n_genera_all=int(combos.size))


def match_fig2_panels(fig_a, fig_b):
    """"""
    ax_a, ax_b = fig_a.axes[0], fig_b.axes[0]
    Wa, Ha = fig_a.get_size_inches(); pa = ax_a.get_position()
    per_unit = pa.height * Ha / np.ptp(ax_a.get_ylim())
    bottom_in, top_in = pa.y0 * Ha, (1 - pa.y1) * Ha
    Hb = bottom_in + top_in + per_unit * np.ptp(ax_b.get_ylim())
    fig_b.set_size_inches(Wa, Hb)
    fig_b.subplots_adjust(left=pa.x0, right=pa.x1, bottom=bottom_in / Hb, top=1 - top_in / Hb)
    fs_a = next((t.get_fontsize() for t in ax_a.texts if t.get_bbox_patch() is not None), None)
    separate_group_labels(fig_b, fontsize=fs_a)
    return Hb


def save_fig2(fig_a, fig_b, out_prefix, dpi=600, composite_dpi=300):
    """"""
    from matplotlib.transforms import Bbox
    from PIL import Image, ImageDraw, ImageFont
    Image.MAX_IMAGE_PIXELS = None
    boxes = []
    for f in (fig_a, fig_b):
        f.canvas.draw()
        boxes.append(f.get_tightbbox(f.canvas.get_renderer()))
    x0 = min(b.x0 for b in boxes) - 0.1; x1 = max(b.x1 for b in boxes) + 0.1
    paths = []
    for tag, f, b in (("a", fig_a, boxes[0]), ("b", fig_b, boxes[1])):
        bb = Bbox([[x0, b.y0 - 0.1], [x1, b.y1 + 0.1]])
        for ext in ("png", "pdf", "svg"):
            f.savefig(f"{out_prefix}{tag}.{ext}", dpi=dpi, bbox_inches=bb)
        paths.append(f"{out_prefix}{tag}.png")
    ims = [Image.open(p).convert("RGB") for p in paths]
    k = composite_dpi / dpi
    ims = [im.resize((round(im.width * k), round(im.height * k)), Image.LANCZOS) for im in ims]
    gap = round(0.25 * composite_dpi)
    C = Image.new("RGB", (max(i.width for i in ims), sum(i.height for i in ims) + gap), "white")
    C.paste(ims[0], (0, 0)); C.paste(ims[1], (0, ims[0].height + gap))
    d = ImageDraw.Draw(C)
    try:
        fnt = ImageFont.truetype("Arial Bold.ttf", round(0.35 * composite_dpi))
    except Exception:
        try:
            fnt = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial Bold.ttf", round(0.35 * composite_dpi))
        except Exception:
            fnt = ImageFont.load_default()
    d.text((10, 5), "a", fill="black", font=fnt); d.text((10, ims[0].height + gap + 5), "b", fill="black", font=fnt)
    C.save(f"{out_prefix}.png", dpi=(composite_dpi, composite_dpi))
    return f"{out_prefix}.png", C.size


def separate_group_labels(fig, fontsize=None, min_gap_pt=None, max_iter=50):
    """Phylum labels (boxed texts at group mid-points) of adjacent one-row groups collide once rows are
    spaced as in panel (a). Push neighbouring labels apart symmetrically until their centres are at least
    `min_gap_pt` apart (default 2.2 x font size), keeping them inside the y-limits. """
    ax = fig.axes[0]
    T = sorted([t for t in ax.texts if t.get_bbox_patch() is not None], key=lambda t: t.get_position()[1])
    if fontsize:
        for t in T: t.set_fontsize(fontsize)
    fs = T[0].get_fontsize() if T else 16
    gap = (min_gap_pt or 2.2 * fs) / 72.0                                   # inches
    per_unit = ax.get_position().height * fig.get_size_inches()[1] / np.ptp(ax.get_ylim())
    g = gap / per_unit                                                      # data units
    y = np.array([t.get_position()[1] for t in T], float)
    lo, hi = ax.get_ylim()[0] + 0.5 * g, ax.get_ylim()[1] - 0.5 * g
    for _ in range(max_iter):
        moved = False
        for i in range(len(y) - 1):
            d = y[i + 1] - y[i]
            if d < g:
                s = (g - d) / 2; y[i] -= s; y[i + 1] += s; moved = True
        y = np.clip(y, lo, hi)
        if not moved: break
    for t, yy in zip(T, y):
        t.set_position((t.get_position()[0], yy))
    return dict(zip([t.get_text() for t in T], np.round(y, 2)))


# ------------------------------------------------ overlap with the biomass indicators (Notes S4, Table S14)
SOIL_CODE = {"Resistant": "resistant", "Permissive": "Permissive"}          # metadata soil codes
S5_SOIL = {"Resistant": "Cereal-dominated (AF193)", "Permissive": "Mixed-system (AF190)"}


def eligible_set(T, kg, soil, prev_min=0.10, abund_min=1e-4):
    """Taxa eligible for TITAN2 / mEQO in one soil (same filter as candidate_pool.R and Table S8):
    present in >= 10% of pots with biomass and mean relative abundance >= 1e-4."""
    md = T[kg]["md"]; X = T[kg]["X"]
    s = md[(md.soil == SOIL_CODE[soil]) & md.Dry_wheat_mass.notna()].index.intersection(X.index)
    Xs = X.loc[s]
    return set(Xs.columns[((Xs > 0).mean() >= prev_min) & (Xs.mean() >= abund_min)])


def indicator_overlap(long, T, table_s5, biomass_up=None):
    """How many consensus-enriched ASVs are also TITAN2 indicators (z+, z-) or mEQO selections.
    """
    from scipy.stats import hypergeom
    from statsmodels.stats.multitest import multipletests
    LB = long.drop_duplicates(["Kingdom", "Soil", "Plant_Species", "Cluster"])
    rows, det = [], []
    for kg in ("bacteria", "fungi"):
        for soil in ("Resistant", "Permissive"):
            E = eligible_set(T, kg, soil)
            calls = LB[(LB.Kingdom == kg) & (LB.Soil == soil)]
            C = set(calls.Cluster); CE = C & E
            cell = table_s5[(table_s5.kingdom == kg) & (table_s5.soil == S5_SOIL[soil])]
            up = (np.mean([(SOILLABEL[soil].capitalize(), w) in biomass_up for w in calls.Plant_Species])
                  if biomass_up is not None else np.nan)
            for lab, S in (("TITAN2 z+", set(cell[cell.TITAN2_direction == "z+"].ASV)),
                           ("TITAN2 z-", set(cell[cell.TITAN2_direction == "z-"].ASV)),
                           ("mEQO selected", set(cell[cell.mEQO == "selected"].ASV))):
                SE = S & E; ov = CE & SE
                exp = len(CE) * len(SE) / len(E)
                p = hypergeom(len(E), len(SE), len(CE)).sf(len(ov) - 1) if SE else np.nan
                rows.append(dict(Kingdom=kg, Soil=SOILLABEL[soil], eligible=len(E), consensus_ASVs=len(C),
                                 consensus_in_eligible=len(CE), indicator_set=lab, set_size=len(S),
                                 set_in_eligible=len(SE), shared=len(C & S), shared_in_eligible=len(ov),
                                 expected=round(exp, 2), fold=round(len(ov) / exp, 2) if exp else np.nan,
                                 p_hypergeometric=p, share_calls_biomass_raising=round(up, 2)))
                for a in sorted(C & S):
                    g = calls[calls.Cluster == a]
                    det.append(dict(Kingdom=kg, Soil=SOILLABEL[soil], indicator_set=lab, ASV=a,
                                    Genus=T[kg]["tax"].Genus.get(a), in_eligible=a in E,
                                    also_mEQO=a in set(cell[cell.mEQO == "selected"].ASV),
                                    n_neighbours=g.Plant_Species.nunique(),
                                    neighbours=", ".join(sorted(set(g.Plant_Species)))))
    s = pd.DataFrame(rows)
    s["q_BH"] = multipletests(s.p_hypergeometric.fillna(1), method="fdr_bh")[1]
    return s, pd.DataFrame(det)


def biomass_raising_neighbours(T, alpha=0.05, control=CONTROL):
    """(soil label, neighbour) pairs whose wheat biomass exceeds wheat x wheat: two-sided Mann-Whitney U,
    uncorrected, as in Table S4 (which this reproduces), and a higher mean."""
    from scipy.stats import mannwhitneyu
    md = T["bacteria"]["md"].dropna(subset=["Dry_wheat_mass"])
    out = set()
    for soil, code in SOIL_CODE.items():
        m = md[md.soil == code]; c = m[m.species == control].Dry_wheat_mass.values
        for w, g in m.groupby("species"):
            if w in (control, "wheat alone"): continue
            x = g.Dry_wheat_mass.values
            if mannwhitneyu(x, c, alternative="two-sided").pvalue < alpha and x.mean() > c.mean():
                out.add((SOILLABEL[soil].capitalize(), w))
    return out

### `bacteria_dotplot_revised` (`bdp`)
Bacterial panel of the genus-level map of the candidate ASVs (Table S3 workbook). Used in 3.1.2.

In [ ]:
%%module bacteria_dotplot_revised bdp
# =============================================================
#  Bacteria genera dot-plot 
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt

# ---- species name corrections -------------------------------------------
# key = the raw name as it appears in the data (lower-cased, spaces);
# value = the correct display name. 
SPECIES_RENAME = {			# from previous mismatch 
    "centaurea cyanus":       "Cyanus segetum",
    "bromus sterilis":        "Anisantha sterilis",
    "festuca arundinacea":    "Lolium arundinaceum",   
    "wheat alone":            "wheat alone",
    "wheat x wheat":          "wheat x wheat",
}
def _skey(raw):
    return str(raw).strip().lower().replace("_", " ")
def format_species_name(raw):
    """Correct known misnamings
    ('daucus carota' -> 'Daucus carota'; hyphen epithets stay lower)."""
    k = _skey(raw)
    if k in SPECIES_RENAME:
        return SPECIES_RENAME[k]
    if k in ("wheat alone", "wheat x wheat"):
        return k
    return k.capitalize()

def create_combined_bacteria_plot(
        filtered_df, cluster_2_genus_d, cluster_2_phylum_d, plant_2_family,
        beneficial_genera_d, function_palette, phyl_col_d,
        default_genus_color="#A9A9A9",
        # -------- editble knobs ---------------------------------------
        fig_width=32, fig_height=18,   # <<< figure size (inches)
        y_fontsize=17,                 # <<< y-axis (genus) label font size
        x_fontsize=22,                 # x-axis (species) label font size
        point_scale=60, point_base=200,# marker size = count*scale + base  (was 30 / 100)
        phylum_gap=1.2,                # distance of phylum labels to the RIGHT of the box
        species_spacing=3,
        out_prefix=None):              # if set, saves png/pdf/svg with this path prefix
    # ---- build genus-level long table ----
    gd = []
    for _, r in filtered_df.iterrows():
        c = r['Cluster']
        g  = cluster_2_genus_d.get(c, "Unknown")
        ph = cluster_2_phylum_d.get(c, "unidentified")
        gd.append({"Plant Species": r['Plant_Species'], "Genus": g, "Phylum": ph,
                   "Function": beneficial_genera_d.get(g, "Unknown"), "Soil": r['Soil']})
    df = pd.DataFrame(gd)

    # ---- correct + de-capitalise species names ----
    df["Species"] = df["Plant Species"].apply(format_species_name)
    df["Family"]  = df["Species"].map(lambda s: plant_2_family.get(s, "Unknown"))

    df_plot = (df.groupby(["Family", "Species", "Genus", "Phylum", "Function", "Soil"])
                 .size().reset_index(name="Cluster_Count"))
    df_plot["Plant_Group"] = df_plot["Family"] + " - " + df_plot["Species"]
    ordered = df_plot.sort_values(["Family", "Species"])["Plant_Group"].unique()

    # ----  first plant starts at position 0 (no empty leading column) ----
    plant_positions = {}
    pos = 0
    for pl in ordered:
        plant_positions[pl] = pos
        pos += species_spacing
    df_plot["x_pos"] = df_plot.apply(
        lambda r: plant_positions[r["Plant_Group"]] - 0.5 if r["Soil"] == "Resistant"
                  else plant_positions[r["Plant_Group"]] + 0.5, axis=1)

    # ---- order genera within phylum ----
    uniq_phyla = sorted(df_plot['Phylum'].unique())
    phyla_genera = {p: [] for p in uniq_phyla}
    for _, r in df_plot.drop_duplicates(['Genus', 'Phylum']).iterrows():
        if r['Genus'] not in phyla_genera[r['Phylum']]:
            phyla_genera[r['Phylum']].append(r['Genus'])
    for p in phyla_genera:
        phyla_genera[p].sort()
    gpp = {}; yp = 0; phylum_positions = {}
    for p in uniq_phyla:
        phylum_positions[p] = yp
        for g in phyla_genera[p]:
            gpp[(g, p)] = yp; yp += 1
    df_plot['y_pos'] = df_plot.apply(lambda r: gpp.get((r['Genus'], r['Phylum']), 0), axis=1)

    # ---- figure size via fig_width/fig_height; use fig,ax ----
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    soil_edge = {"Resistant": "#404040", "Permissive": "#A0A0A0"}
    for soil, edge in soil_edge.items():
        sd = df_plot[df_plot["Soil"] == soil]
        colors = [function_palette.get(f, default_genus_color) for f in sd["Function"]]
        ax.scatter(sd["x_pos"], sd["y_pos"],
                   s=sd["Cluster_Count"] * point_scale + point_base,   # <<< FIX (2): marker size
                   c=colors, marker='o', edgecolor=edge, linewidth=2, alpha=0.85, label=soil)

    # y ticks
    yt, ytl = [], []
    for p in uniq_phyla:
        for g in phyla_genera[p]:
            yt.append(gpp[(g, p)]); ytl.append(g)
    ax.set_yticks(yt); ax.set_yticklabels(ytl, fontsize=y_fontsize, fontstyle='italic')
    ax.set_ylabel("Bacterial Genus", fontsize=y_fontsize + 7)

    # x ticks — corrected names, one per plant, italic
    xt  = [plant_positions[pl] for pl in ordered]
    xtl = [pl.split(" - ")[1] for pl in ordered]
    ax.set_xticks(xt); ax.set_xticklabels(xtl, rotation=90, fontsize=x_fontsize, fontstyle='italic')

    # phylum band separators
    for p in uniq_phyla[1:]:
        ax.axhline(phylum_positions[p] - 0.5, color='gray', lw=2, alpha=0.7)

    # ----  set limits first, then place phylum labels just OUTSIDE the box ----
    xmax = max(plant_positions.values())
    ax.set_xlim(-species_spacing * 0.6, xmax + species_spacing * 0.6)  # tight left edge
    ax.set_ylim(-1, yp)
    x_right = ax.get_xlim()[1]
    for p in uniq_phyla:
        n = len(phyla_genera[p])
        if n == 0:
            continue
        centre = phylum_positions[p] + n / 2 - 0.5
        ax.text(x_right + phylum_gap, centre, p, fontsize=y_fontsize - 1, fontweight='bold',
                va='center', ha='left', clip_on=False,
                bbox=dict(facecolor=phyl_col_d.get(p, "#7f7f7f"), alpha=0.2,
                          edgecolor=phyl_col_d.get(p, "#7f7f7f"), boxstyle="round,pad=0.5"))

    # family separators
    cur = None
    for pl in ordered:
        fam = pl.split(' - ')[0]
        if fam != cur:
            ax.axvline(plant_positions[pl] - 1.5, color='gray', lw=2, alpha=0.6); cur = fam

    # legends (marker size in legend bumped to match)
    soil_leg = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
                   markeredgecolor='#404040', markeredgewidth=2, markersize=13, label='Cereal-dominated Soil'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
                   markeredgecolor='#A0A0A0', markeredgewidth=2, markersize=13, label='Mixed-system Soil')]
    func_leg = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=13, label=f)
                for f, c in function_palette.items() if f in df_plot["Function"].values]
    if "Unknown" in df_plot["Function"].values:
        func_leg.append(plt.Line2D([0], [0], marker='o', color='w',
                        markerfacecolor=default_genus_color, markersize=13, label="Unknown Function"))
    ax.legend(handles=soil_leg + func_leg, bbox_to_anchor=(1.10, 1), loc="upper left",
              title="Soil Type & Function", title_fontsize=16, fontsize=13, frameon=True)

    ax.grid(axis='y', linestyle='--', alpha=0.3)
    fig.subplots_adjust(right=0.82, bottom=0.22, top=0.96)

    if out_prefix:
        fig.savefig(out_prefix + ".png", dpi=600, bbox_inches='tight')
        fig.savefig(out_prefix + ".pdf", bbox_inches='tight')
        fig.savefig(out_prefix + ".svg", bbox_inches='tight')
        df_plot.to_csv(out_prefix + ".csv", index=False)
    return fig, df_plot

### `fungi_dotplot_cell` (`fdp`)
Fungal panel of the same map. Used in 3.1.2.

In [ ]:
%%module fungi_dotplot_cell fdp
# ============================================================================
#  Fungal genera dot-plot — revised, parameterised (single notebook cell)
#  Matches the bacteria figure: same y-axis spacing, same fixes.
#  Fixes: (1) figure-size & (2) y-axis-size knobs, (3) larger markers,
#         (4) no empty leading column, (5) phylum labels outside the box,
#         (6) corrected + de-capitalised species names on the x-axis.
#  Coloured by PHYLUM (fungal convention)
#  Assumes these already exist in notebook:
#     filtered_df, cluster_2_genus_d, cluster_2_phylum_d, plant_2_family
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

# fungal phylum palette (edit colours here if you like)
FUNGI_PHYL_COL = {
    "Ascomycota":"#87CEEB","Basidiomycota":"#32CD32","Mortierellomycota":"#9370DB",
    "Glomeromycota":"#8B4513","Mucoromycota":"#228B22","Chytridiomycota":"#FF69B4",
    "Rozellomycota":"#BA55D3","Olpidiomycota":"#FFD700","unidentified":"#A9A9A9",
    "Multi-affiliation":"#000000",
}

# ---- species-name corrections (key = raw name in YOUR data, lower-cased) ----
SPECIES_RENAME = {
    "centaurea cyanus":    "Cyanus segetum",
    "bromus sterilis":     "Anisantha sterilis",
    "festuca arundinacea": "Lolium arundinaceum",   # only if present in your data
    "wheat alone":         "wheat alone",
    "wheat x wheat":       "wheat x wheat",
}
def _skey(raw):
    return str(raw).strip().lower().replace("_", " ")
def format_species_name(raw):
    """Correct known misnamings; else capitalise first letter only
       ('daucus carota' -> 'Daucus carota'; hyphen epithets stay lower)."""
    k = _skey(raw)
    if k in SPECIES_RENAME:                    return SPECIES_RENAME[k]
    if k in ("wheat alone", "wheat x wheat"):  return k
    return k.capitalize()

def create_combined_fungi_plot(
        multi_method_data, cluster_2_genus_d, cluster_2_phylum_d, plant_2_family,
        phyl_col_d=None,
        # -------------------- TUNABLE KNOBS ---------------------------------
        fig_width=32, fig_height=18,     # (1) figure size, inches
        y_fontsize=17,                   # (2) y-axis (genus) label font size
        x_fontsize=20,                   # x-axis (species) label font size
        point_scale=100, point_base=150, # (3) marker size = count*scale + base (orig 100 / 90)
        phylum_gap=1.2,                  # (5) phylum-label distance right of box
        species_spacing=3,
        legend_x=1.13,                   # legend distance right of box (raise if labels collide)
        out_prefix=None):                # if set: saves png(600dpi)/pdf/svg/csv
    if phyl_col_d is None:
        phyl_col_d = FUNGI_PHYL_COL

    # family resolver keyed on RAW species name (independent of display rename)
    fam_by_key = {_skey(k): v for k, v in plant_2_family.items()}

    gd = []
    for _, r in multi_method_data.iterrows():
        c = r['Cluster']
        gd.append({"raw":    r['Plant_Species'],
                   "Genus":  cluster_2_genus_d.get(c, "Unknown"),
                   "Phylum": cluster_2_phylum_d.get(c, "unidentified"),
                   "Soil":   r['Soil']})
    df = pd.DataFrame(gd)

    # (6) correct + de-capitalise species names; resolve family from raw key
    df["Species"] = df["raw"].apply(format_species_name)
    df["Family"]  = df["raw"].apply(lambda s: fam_by_key.get(_skey(s), "Unknown"))
    miss = df.loc[df["Family"] == "Unknown", "raw"].unique()
    if len(miss):
        print("WARNING: no family mapping for:", list(miss))

    df_plot = (df.groupby(["Family","Species","Genus","Phylum","Soil"])
                 .size().reset_index(name="Cluster_Count"))
    df_plot["Plant_Group"] = df_plot["Family"] + " - " + df_plot["Species"]
    ordered = df_plot.sort_values(["Family","Species"])["Plant_Group"].unique()

    # (4) first plant at position 0 -> no empty leading column
    plant_positions, pos = {}, 0
    for pl in ordered:
        plant_positions[pl] = pos; pos += species_spacing
    df_plot["x_pos"] = df_plot.apply(
        lambda r: plant_positions[r["Plant_Group"]] - 0.5 if r["Soil"] == "Resistant"
                  else plant_positions[r["Plant_Group"]] + 0.5, axis=1)

    # genera ordered within phylum (same y-spacing logic as bacteria fig)
    uniq_phyla = sorted(df_plot['Phylum'].unique())
    phyla_genera = {p: [] for p in uniq_phyla}
    for _, r in df_plot.drop_duplicates(['Genus','Phylum']).iterrows():
        if r['Genus'] not in phyla_genera[r['Phylum']]:
            phyla_genera[r['Phylum']].append(r['Genus'])
    for p in phyla_genera: phyla_genera[p].sort()
    gpp, yp, phylum_positions = {}, 0, {}
    for p in uniq_phyla:
        phylum_positions[p] = yp
        for g in phyla_genera[p]:
            gpp[(g, p)] = yp; yp += 1
    df_plot['y_pos'] = df_plot.apply(lambda r: gpp.get((r['Genus'], r['Phylum']), 0), axis=1)

    # (1) figure size
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    soil_edge = {"Resistant": "#404040", "Permissive": "#A0A0A0"}
    for soil, edge in soil_edge.items():
        sd = df_plot[df_plot["Soil"] == soil]
        ax.scatter(sd["x_pos"], sd["y_pos"],
                   s=sd["Cluster_Count"] * point_scale + point_base,   # (3) marker size
                   c=[phyl_col_d.get(p, "#7f7f7f") for p in sd["Phylum"]],
                   marker='o', edgecolor=edge, linewidth=2, alpha=0.8, label=soil)

    # y ticks
    yt, ytl = [], []
    for p in uniq_phyla:
        for g in phyla_genera[p]:
            yt.append(gpp[(g, p)]); ytl.append(g)
    ax.set_yticks(yt); ax.set_yticklabels(ytl, fontsize=y_fontsize, fontstyle='italic')
    ax.set_ylabel("Fungal Genus", fontsize=y_fontsize + 7)

    # x ticks — corrected names, italic
    xt  = [plant_positions[pl] for pl in ordered]
    xtl = [pl.split(" - ")[1] for pl in ordered]
    ax.set_xticks(xt); ax.set_xticklabels(xtl, rotation=90, fontsize=x_fontsize, fontstyle='italic')

    # phylum band separators
    for p in uniq_phyla[1:]:
        ax.axhline(phylum_positions[p] - 0.5, color='gray', lw=2, alpha=0.7)

    # (5) limits first, then phylum labels just OUTSIDE the box
    xmax = max(plant_positions.values())
    ax.set_xlim(-species_spacing * 0.6, xmax + species_spacing * 0.6)
    ax.set_ylim(-1, yp)
    x_right = ax.get_xlim()[1]
    for p in uniq_phyla:
        n = len(phyla_genera[p])
        if n == 0: continue
        centre = phylum_positions[p] + n / 2 - 0.5
        ax.text(x_right + phylum_gap, centre, p, fontsize=y_fontsize, fontweight='bold',
                va='center', ha='left', clip_on=False,
                bbox=dict(facecolor=phyl_col_d.get(p, "#7f7f7f"), alpha=0.2,
                          edgecolor=phyl_col_d.get(p, "#7f7f7f"), boxstyle="round,pad=0.5"))

    # family separators
    cur = None
    for pl in ordered:
        fam = pl.split(' - ')[0]
        if fam != cur:
            ax.axvline(plant_positions[pl] - 1.5, color='gray', lw=2, alpha=0.6); cur = fam

    # legend (push further right than bacteria fig so it clears phylum labels)
    soil_leg = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
                   markeredgecolor='#404040', markeredgewidth=2, markersize=13, label='Cereal-dominated Soil'),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
                   markeredgecolor='#A0A0A0', markeredgewidth=2, markersize=13, label='Mixed-system Soil')]
    phyl_leg = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=13, label=p)
                for p, c in phyl_col_d.items() if p in uniq_phyla]
    ax.legend(handles=soil_leg + phyl_leg, bbox_to_anchor=(legend_x, 1), loc="upper left",
              title="Soil Type & Phylum", title_fontsize=16, fontsize=13, frameon=True)

    ax.grid(axis='y', linestyle='--', alpha=0.3)
    fig.subplots_adjust(right=0.80, bottom=0.22, top=0.96)

    if out_prefix:
        fig.savefig(out_prefix + ".png", dpi=600, bbox_inches='tight')
        fig.savefig(out_prefix + ".pdf", bbox_inches='tight')
        fig.savefig(out_prefix + ".svg", bbox_inches='tight')
        df_plot.to_csv(out_prefix + ".csv", index=False)
    return fig, df_plot

### `biomass_tests` (`bt`)
Wheat-biomass tests: ANOVA, scale check, between-soil tests, legume test, Table S4. Used in 3.2.1.

In [ ]:
%%module biomass_tests bt
# -*- coding: utf-8 -*-
"""Wheat biomass statistics for Results 3.2.1 (Fig. 3, Table S3, Table S5 text), called from Paper_figures.ipynb.

  anova_tables(meta)        two-way ANOVA (soil x neighbour) on log biomass, all treatments and weeds only,
                            residual checks on raw and log scales
  between_soils(meta)       Mann-Whitney between soils for each of the 30 treatments, BH across the 30 (Fig. 3 asterisks)
  legume_test(meta)         species-mean wheat biomass under the four legumes vs the 24 other weeds
  table_s3(meta, out_xlsx)  each neighbour x soil against wheat x wheat and wheat alone: n, mean +/- SD,
                            difference with bootstrap 95% CI, two-sided Mann-Whitney P, BH q across the 56 contrasts
Soil codes: 'resistant' = cereal-dominated (AF193), 'Permissive' = mixed-system (AF190)."""
import numpy as np, pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

SOIL = {"resistant": "Cereal-dominated (AF193)", "Permissive": "Mixed-system (AF190)"}
CTRL = ["wheat x wheat", "wheat alone"]
LEGUMES = ["Lotus corniculatus", "Medicago lupulina", "Trigonella foenum-graecum", "Vicia cracca"]
SYN = {"Centaurea cyanus": "Cyanus segetum", "Festuca arundinacea": "Lolium arundinaceum", "Bromus sterilis": "Anisantha sterilis"}


def load(meta_csv):
    m = pd.read_csv(meta_csv).dropna(subset=["Dry_wheat_mass"]).copy()
    m["species"] = m.species.replace(SYN)
    m["logB"] = np.log(m.Dry_wheat_mass)
    return m


def _aov(d, y):
    f = smf.ols(f"{y} ~ C(soil) * C(species)", data=d).fit()
    a = sm.stats.anova_lm(f, typ=2)
    res = f.resid
    return a, dict(shapiro_W=stats.shapiro(res).statistic,
                   brown_forsythe_P=stats.levene(*[g[y].values for _, g in d.groupby(["soil", "species"])], center="median").pvalue)


def anova_tables(m):
    rows = []
    for set_, d in (("all treatments", m), ("weeds only", m[~m.species.isin(CTRL)])):
        for y in ("Dry_wheat_mass", "logB"):
            a, chk = _aov(d, y)
            for term in ("C(soil)", "C(species)", "C(soil):C(species)"):
                rows.append(dict(set=set_, scale="raw" if y == "Dry_wheat_mass" else "log", term=term.replace("C(", "").replace(")", ""),
                                 df=int(a.loc[term, "df"]), df_resid=int(a.loc["Residual", "df"]), F=a.loc[term, "F"], P=a.loc[term, "PR(>F)"], **chk))
    return pd.DataFrame(rows)


def between_soils(m):
    rows = []
    for sp, g in m.groupby("species"):
        a, b = g[g.soil == "resistant"].Dry_wheat_mass, g[g.soil == "Permissive"].Dry_wheat_mass
        rows.append(dict(species=sp, n_cereal=len(a), n_mixed=len(b), mean_cereal=a.mean(), mean_mixed=b.mean(),
                         P=stats.mannwhitneyu(a, b, alternative="two-sided").pvalue))
    t = pd.DataFrame(rows); t["q_BH"] = multipletests(t.P, method="fdr_bh")[1]
    return t.sort_values("P")


def legume_test(m):
    s = m[~m.species.isin(CTRL)].groupby(["soil", "species"]).Dry_wheat_mass.mean().reset_index()
    s["legume"] = s.species.isin(LEGUMES)
    rows = []
    for sc, g in s.groupby("soil"):
        rows.append(dict(soil=SOIL[sc], n_legumes=int(g.legume.sum()), n_other=int((~g.legume).sum()),
                         median_legumes=g[g.legume].Dry_wheat_mass.median(), median_other=g[~g.legume].Dry_wheat_mass.median(),
                         P=stats.mannwhitneyu(g[g.legume].Dry_wheat_mass, g[~g.legume].Dry_wheat_mass, alternative="two-sided").pvalue))
    f = smf.ols("Dry_wheat_mass ~ C(soil) + legume", data=s).fit()
    rows.append(dict(soil="both (linear model with soil)", n_legumes=int(s.legume.sum()), n_other=int((~s.legume).sum()),
                     median_legumes=s[s.legume].Dry_wheat_mass.median(), median_other=s[~s.legume].Dry_wheat_mass.median(),
                     P=f.pvalues["legume[T.True]"]))
    return pd.DataFrame(rows)


def _boot_ci(x, y, B=4999, seed=1):
    r = np.random.default_rng(seed)
    d = [r.choice(x, len(x)).mean() - r.choice(y, len(y)).mean() for _ in range(B)]
    return np.percentile(d, [2.5, 97.5])


def table_s3(m, out_xlsx=None):
    rows = []
    for sc in ("resistant", "Permissive"):
        b = m[m.soil == sc]; ww = b[b.species == "wheat x wheat"].Dry_wheat_mass.values; al = b[b.species == "wheat alone"].Dry_wheat_mass.values
        for sp, g in b.groupby("species"):
            x = g.Dry_wheat_mass.values; r = dict(soil=SOIL[sc], species=sp, n=len(x), mean_g=x.mean(), sd_g=x.std(ddof=1))
            if sp not in CTRL:
                lo, hi = _boot_ci(x, ww); lo2, hi2 = _boot_ci(x, al)
                r.update(delta_vs_wxw_g=x.mean() - ww.mean(), ci_lo_wxw=lo, ci_hi_wxw=hi,
                         P_vs_wxw=stats.mannwhitneyu(x, ww, alternative="two-sided").pvalue,
                         delta_vs_alone_g=x.mean() - al.mean(), ci_lo_alone=lo2, ci_hi_alone=hi2,
                         P_vs_alone=stats.mannwhitneyu(x, al, alternative="two-sided").pvalue)
            rows.append(r)
    t = pd.DataFrame(rows); k = t.P_vs_wxw.notna()
    t.loc[k, "q_vs_wxw"] = multipletests(t.loc[k, "P_vs_wxw"], method="fdr_bh")[1]
    t.loc[k, "q_vs_alone"] = multipletests(t.loc[k, "P_vs_alone"], method="fdr_bh")[1]
    t["is_control"] = ~k; t = t.sort_values(["soil", "is_control", "mean_g"], ascending=[True, False, False]).drop(columns="is_control")
    if out_xlsx:
        readme = pd.DataFrame({"Table S3": [
            "Wheat aboveground biomass under each neighbour, per soil (sequenced pots).",
            "Δ, difference in mean biomass (g) from the wheat × wheat control (W×W) or from wheat alone, with bootstrap 95% CI (4,999 resamples).",
            "P, two-sided Mann–Whitney U; q, Benjamini–Hochberg across the 56 neighbour × soil contrasts, separately for each control."]})
        with pd.ExcelWriter(out_xlsx) as w:
            readme.to_excel(w, sheet_name="README", index=False); t.round(4).to_excel(w, sheet_name="Table S3", index=False)
    return t


def scale_check(m):
    """Why log scale (Methods 2.7, Methods S3): Spearman correlation between group mean and SD of wheat biomass
    across the treatment x soil groups, on the raw and log scales."""
    rows = []
    for lab, v in (("raw", m.Dry_wheat_mass), ("log", np.log(m.Dry_wheat_mass))):
        g = v.groupby([m.soil, m.species]).agg(["mean", "std"])
        r = stats.spearmanr(g["mean"], g["std"])
        rows.append(dict(scale=lab, groups=len(g), spearman_rho_mean_sd=r[0], P=r[1]))
    return pd.DataFrame(rows)

### `control_size` (`csz`)
Control-size check: runs `pooled_check.R` and `vs_alone.R`, classifies the Fig. 2 calls, Table S3 sheets. Used in 3.1.2.

In [ ]:
%%module control_size csz
# -*- coding: utf-8 -*-
"""Control-size check for Fig. 2 (Methods S3, Table S3), called in Paper_figures.ipynb 3.1.2.
"""
import os, subprocess, shutil
import pandas as pd


def run(matrix_dir, fig2_values_csv, out_dir, rscript=None):
    rscript = rscript or shutil.which("Rscript") or "Rscript"
    r = subprocess.run([rscript, os.path.join(os.environ.get("PAPER_CODE", os.getcwd()), "pooled_check.R"),   # [I7] was "pooled_check.R" (relative to the working folder)
                        matrix_dir, fig2_values_csv, out_dir], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr[-2000:])
    r = subprocess.run([rscript, os.path.join(os.environ.get("PAPER_CODE", os.getcwd()), "vs_alone.R"),   # [I7] was "vs_alone.R" (relative to the working folder)
                        matrix_dir, out_dir], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr[-2000:])
    return pd.read_csv(os.path.join(out_dir, "pooled_weeds_vs_wxw.csv")), pd.read_csv(os.path.join(out_dir, "fig2_neighbour_vs_other_weeds.csv"))


def classify(fig2, pooled, spec, alone=None):
    k = ["kingdom", "soil", "ASV"]
    f = fig2.merge(pooled[k + ["n_methods"]], on=k, how="left")
    f["called_all_weeds_vs_WxW"] = f.n_methods.fillna(0) >= 2
    f = f.merge(spec[k + ["neighbour", "DESeq2_l", "DESeq2_q", "edgeR_l", "edgeR_q"]], on=k + ["neighbour"], how="left")
    f["specific_both_methods"] = (f.DESeq2_q < 0.05) & (f.edgeR_q < 0.05)
    f["specific_one_method"] = (f.DESeq2_q < 0.05) | (f.edgeR_q < 0.05)
    f = f.drop(columns="n_methods")
    if alone is not None:
        a = alone[k + ["neighbour", "n_methods", "DESeq2_l"]].rename(columns={"n_methods": "methods_vs_alone", "DESeq2_l": "DESeq2_l_vs_alone"})
        f = f.merge(a, on=k + ["neighbour"], how="left"); f["enriched_vs_wheat_alone"] = f.methods_vs_alone.fillna(0) >= 2
    return f


def summary(calls):
    a = calls.drop_duplicates(["kingdom", "soil", "ASV"])
    s = dict(calls=len(calls), ASVs=len(a), ASVs_called_all_weeds=int(a.called_all_weeds_vs_WxW.sum()),
             calls_specific_both=int(calls.specific_both_methods.sum()), calls_specific_one=int(calls.specific_one_method.sum()))
    if "enriched_vs_wheat_alone" in calls:
        v = calls[calls.enriched_vs_wheat_alone]
        s.update(calls_vs_alone=len(v), ASVs_vs_alone=v.drop_duplicates(["kingdom", "soil", "ASV"]).shape[0],
                 share_positive_vs_alone=round(float((calls.DESeq2_l_vs_alone > 0).mean()), 2))
    return s


def table_s3(enriched_xlsx, calls, pooled, out_xlsx):
    x = pd.read_excel(enriched_xlsx, sheet_name=None)
    readme = x.pop("README"); readme.columns = ["Table S3"]
    readme.loc[len(readme)] = ["Control-size check (Methods S3): control_size_calls, each Fig. 2 call against the other 27 weeds pooled "
                               "(DESeq2, edgeR; one-sided; Benjamini-Hochberg within kingdom x soil) and whether its ASV is called when all "
                               "weeds are pooled against wheat x wheat, and (enriched_vs_wheat_alone) whether it is also enriched relative to wheat alone with the "
                               "Fig. 2 rule; pooled_weeds_vs_WxW, ASVs called by at least one method in that contrast. metagenomeSeq P values are "
                               "Bonferroni-adjusted, DESeq2 and edgeR Benjamini-Hochberg."]
    c = calls.copy(); p = pooled[pooled.n_methods >= 1].copy()
    for d in (c, p): d["ASV"] = d.ASV.str.replace("Cluster_", "ASV_")
    with pd.ExcelWriter(out_xlsx) as w:
        readme.to_excel(w, sheet_name="README", index=False)
        for k, v in x.items(): v.to_excel(w, sheet_name=k, index=False)
        c.round(4).to_excel(w, sheet_name="control_size_calls", index=False)
        p.round(4).to_excel(w, sheet_name="pooled_weeds_vs_WxW", index=False)
    return out_xlsx

### `fig_alpha_diversity` (`fa`)
Α-diversity table and Fig. S5. Used in 3.1.1.

In [ ]:
%%module fig_alpha_diversity fa
"""
Fig. S15 -- weed effects on microbial alpha-diversity.

Observed richness and Shannon diversity per pot, summarised per neighbour
species and soil, for both kingdoms. Produces Fig. S15 and the accompanying
two-way ANOVA table.

Inputs (paper_1/data/matrice_phase1/df/):
  otu_bacteria_table_R_1314ASVs.csv   TAB-separated despite the .csv extension
  otu_fungi_table_R_647ASVs.csv       same
  metadata_bacteria_1314ASVs.csv
  metadata_fungi_647ASVs.csv

"""

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

DATA_DIR = os.path.join("paper_1", "data", "matrice_phase1", "df")
OUT_DIR = "results_meqo_convergence"

C_CER = "#B5651D"      # AF193 cereal-dominated  -- same pair as Fig. 4b
C_MIX = "#2E6FA7"      # AF190 mixed-system
META_GREY = "#8B949D"
CONTROLS = {"wheat alone", "wheat x wheat", "Wheat Alone", "Wheat x Wheat"}
SOIL_LABEL = {"resistant": "cereal", "Permissive": "mixed"}

# resistant = cereal-dominated (AF193); Permissive = mixed-system (AF190).
AF = {"resistant": "AF193", "Permissive": "AF190"}

# Superseded binomials in the bacterial metadata -> the accepted names in Table S1
SYNONYMS = {"Centaurea cyanus": "Cyanus segetum",
            "Festuca arundinacea": "Lolium arundinaceum",
            "Bromus sterilis": "Anisantha sterilis"}


def _frame(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_linewidth(0.8)
        ax.spines[sp].set_color("#3A3B38")


def load_alpha(data_dir=DATA_DIR):
    """Per-pot richness and Shannon for both kingdoms.

    sep=None + engine='python' sniffs the delimiter, so this works whether the
    OTU tables are tab- or comma-separated (they are tab-separated, despite the
    extension -- reading them with the default comma gives a single column and
    a silently degenerate result).
    """
    out = {}
    for kingdom, otu_f, meta_f in [
            ("bacteria", "otu_bacteria_table_R_1314ASVs.csv",
             "metadata_bacteria_1314ASVs.csv"),
            ("fungi", "otu_fungi_table_R_647ASVs.csv",
             "metadata_fungi_647ASVs.csv")]:
        otu = pd.read_csv(os.path.join(data_dir, otu_f), sep=None,
                          engine="python", index_col=0)
        meta = pd.read_csv(os.path.join(data_dir, meta_f), sep=None,
                           engine="python")
        samples = [s for s in meta["sample"] if s in otu.columns]
        M = otu[samples].T
        d = meta.set_index("sample").loc[samples].copy()
        d["richness"] = (M > 0).sum(1).values
        p = M.div(M.sum(1), axis=0).values
        with np.errstate(divide="ignore", invalid="ignore"):
            d["shannon"] = -np.nansum(np.where(p > 0, p * np.log(p), 0), axis=1)
        d["species"] = d["species"].replace(SYNONYMS)   # harmonise to Table S1
        d["soil2"] = d["soil"].map(SOIL_LABEL)
        d["is_control"] = d["species"].isin(CONTROLS)
        out[kingdom] = d.reset_index()
    return out


def alpha_anova(d, resp="richness"):
    """Two-way ANOVA with omega-squared effect sizes.

    Sequential (Type I) sums of squares, which is what the manuscript reports
    for the PERMANOVA and is kept consistent here. omega2 rather than eta2
    because it is the less biased estimator for a fixed-effects design and is
    what the text quotes.
    """
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    m = ols(f"{resp} ~ C(soil) * C(species)", data=d).fit()
    a = sm.stats.anova_lm(m, typ=1)
    ms_resid = a.loc["Residual", "sum_sq"] / a.loc["Residual", "df"]
    sst = a["sum_sq"].sum()
    a["omega2"] = [(r.sum_sq - r.df * ms_resid) / (sst + ms_resid)
                   if i != "Residual" else np.nan for i, r in a.iterrows()]
    return a


def species_means(d, resp="richness"):
    g = (d.groupby(["species", "soil2"])[resp]
           .agg(["mean", "std", "size"]).reset_index())
    g["se"] = g["std"] / np.sqrt(g["size"])
    return g


def fig_alpha_diversity(alpha, resp="richness", out_dir=OUT_DIR,
                        fname="FigS15_alpha_diversity.png",
                        width_in=7.2, height_in=7.4, scale=1.0, dpi=400):
    """Richness per neighbour species and soil, both kingdoms.
    """
    s = scale
    FS = {"base": 9 * s, "ann": 8 * s, "tick": 7 * s, "letter": 11 * s}
    fig, axes = plt.subplots(2, 1, figsize=(width_in, height_in),
                             gridspec_kw=dict(hspace=0.78, left=0.115,
                                              right=0.995, top=0.955,
                                              bottom=0.155))
    ylab = ("Observed richness (ASVs)" if resp == "richness"
            else "Shannon diversity")
    for ax, (kingdom, nm, marker) in zip(
            axes, [("bacteria", "Bacteria", "rpoB"), ("fungi", "Fungi", "ITS1")]):
        g = species_means(alpha[kingdom], resp)
        weeds = g[~g["species"].isin(CONTROLS)]
        order = (weeds[weeds.soil2 == "mixed"].set_index("species")["mean"]
                 .reindex(weeds["species"].unique()).sort_values().index.tolist())
        xpos = {sp: i for i, sp in enumerate(order)}
        for ctrl, ls in [("wheat alone", "--"), ("wheat x wheat", ":")]:
            for so, c in [("cereal", C_CER), ("mixed", C_MIX)]:
                row = g[(g["species"] == ctrl) & (g.soil2 == so)]
                if len(row):
                    ax.axhline(row["mean"].iloc[0], color=c, lw=0.8 * s,
                               ls=ls, alpha=0.55, zorder=1)
        for so, c, lab in [("cereal", C_CER, "Cereal-dominated (AF193)"),
                           ("mixed", C_MIX, "Mixed-system (AF190)")]:
            sub = (g[g.soil2 == so].set_index("species")
                   .reindex(order).reset_index())
            ax.errorbar([xpos[sp] for sp in sub["species"]], sub["mean"],
                        yerr=sub["se"], fmt="o", ms=3.4 * s, lw=0,
                        elinewidth=0.9 * s, capsize=0, color=c, zorder=3,
                        label=lab)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=90, fontsize=FS["tick"],
                           style="italic")
        ax.set_ylabel(ylab, fontsize=FS["base"])
        ax.set_title(f"{nm} ({marker}): neighbour identity shifts richness, "
                     f"and the shift differs by soil",
                     fontsize=FS["base"], loc="left", pad=6 * s)
        ax.margins(x=0.02, y=0.06)
        _frame(ax)
        ax.tick_params(labelsize=FS["tick"])
    axes[0].legend(frameon=False, fontsize=FS["ann"], loc="upper left",
                   handlelength=0.9, handletextpad=0.45, borderpad=0)
    axes[0].annotate("dashed: wheat alone   dotted: wheat \u00d7 wheat",
                     xy=(0.985, 0.03), xycoords="axes fraction", ha="right",
                     va="bottom", fontsize=FS["ann"] - 0.6, color=META_GREY)
    for ax, L in zip(axes, "ab"):
        bb = ax.get_position()
        fig.text(max(0.004, bb.x0 - 0.085), bb.y1 + 0.028, L,
                 fontsize=FS["letter"], fontweight="bold", ha="left", va="top")
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, fname), dpi=dpi, bbox_inches="tight",
                facecolor="white")
    return fig


def alpha_table(alpha, out_dir=OUT_DIR, fname="alpha_diversity_anova.csv"):
    """ANOVA terms for both kingdoms and both responses."""
    rows = []
    for kingdom in ("bacteria", "fungi"):
        for resp in ("richness", "shannon"):
            a = alpha_anova(alpha[kingdom], resp)
            for term, r in a.iterrows():
                rows.append(dict(kingdom=kingdom, response=resp,
                                 term=term.replace("C(", "").replace(")", ""),
                                 df=int(r["df"]), F=r.get("F"),
                                 p=r.get("PR(>F)"), omega2=r.get("omega2")))
    t = pd.DataFrame(rows)
    os.makedirs(out_dir, exist_ok=True)
    t.to_csv(os.path.join(out_dir, fname), index=False)
    return t

## 3.1.1 Community composition and assembly

### Fig. 1; Fig. S5 (α-diversity); Fig. S6 (Glomeromycota); Fig. S7 (bacterial assembly metrics); turnover share

In [ ]:
fig1 = mf.fig1(MATRIX_DIR, os.path.join(FIG, "Fig1"), mf.table1_from_csv(os.path.join(RESULTS, "Table1_variance_partition.csv")))
alpha = fa.load_alpha(data_dir=MATRIX_DIR); fa.alpha_table(alpha, out_dir=TAB)
fa.fig_alpha_diversity(alpha, out_dir=FIG, fname="FigS5_alpha_diversity.png"); plt.close("all")
figS6, glom = sf.fig_glomeromycota(MATRIX_DIR, os.path.join(FIG, "FigS6_glomeromycota"))
figS7 = sf.fig_assembly_bacteria(NST_DIR, MATRIX_DIR, os.path.join(FIG, "FigS7_assembly_bacteria"))
turn = sf.turnover_table(MATRIX_DIR); turn.to_csv(os.path.join(TAB, "turnover_share.csv"), index=False)
print(glom.round(2)); turn

                           mean  median  size
soilL                                        
Cereal-dominated (AF193)   7.71    1.50   222
Mixed-system (AF190)      14.34    3.55   227


,kingdom,soil,group,n,turnover,nestedness,total,pct_turnover
0,bacteria,Cereal-dominated (AF193),All pots,223,0.563801,0.075923,0.639725,88.131894
1,bacteria,Cereal-dominated (AF193),Poaceae,46,0.525195,0.081612,0.606807,86.550653
2,bacteria,Cereal-dominated (AF193),Fabaceae,29,0.559603,0.065363,0.624967,89.541285
3,bacteria,Cereal-dominated (AF193),Papaveraceae,7,0.585621,0.049697,0.635318,92.177568
4,bacteria,Cereal-dominated (AF193),Ranunculaceae,6,0.520995,0.105019,0.626014,83.224143
...,...,...,...,...,...,...,...,...
63,fungi,Mixed-system (AF190),Apiaceae,8,0.633376,0.100426,0.733802,86.314234
64,fungi,Mixed-system (AF190),Boraginaceae,5,0.656893,0.096514,0.753407,87.189664
65,fungi,Mixed-system (AF190),Juncaceae,6,0.624185,0.063776,0.687961,90.729726
66,fungi,Mixed-system (AF190),Wheat × wheat,9,0.627499,0.064669,0.692168,90.657039


## 3.1.2 Neighbour-enriched taxa

### Fig. 2, Table S3; Methods S3 method agreement and cross-family convergence

`da_core_check()`: Fig. 2, the candidate-ASV workbook (Table S3), method agreement and cross-family convergence (Methods S3).   <!-- [I2] formerly da_core_check.py -->

In [ ]:
def da_core_check(MATRIX, OUT, ROOTS):
    """Differential abundance for Results 3.1.2: Fig. 2, the enriched-ASV workbook (Table S3) and the Methods S3 tests.
    A call needs enrichment under the neighbour (log2 fold change > 2) by at least two of DESeq2, edgeR and metagenomeSeq:
    at adjusted P < 0.05 for Fig. 2 (DESeq2, edgeR BH; metagenomeSeq Bonferroni), at nominal method-specific thresholds for the broader screen (Table S3).
    Methods S3: method agreement against chance (hypergeometric) and cross-family convergence (999 permutations).
    """
    import os, sys, io, contextlib
    import numpy as np, pandas as pd
    from scipy.stats import hypergeom
    import da_consensus as dc

    os.makedirs(OUT, exist_ok=True)
    TT = dc.load_tables(MATRIX)
    rng = np.random.default_rng(1)


    def calls_full(kg, soil):
        out = {}
        for m, (sub, pat) in dc.FILES.items():
            f = next(os.path.join(r, sub, pat.format(kg=kg, soil=soil)) for r in ROOTS
                     if os.path.exists(os.path.join(r, sub, pat.format(kg=kg, soil=soil))))
            raw = pd.read_csv(f); df = dc.read_export(f, m)
            df["q"] = raw.padj.values if m == "DESeq2" else (raw.FDR.values if m == "edgeR" else df.p)   # metagenomeSeq p is adjusted
            df = df[df.weed != "wheat alone"].copy()
            sign, _, _ = dc.orient_and_check(TT, kg, soil, df, os.path.basename(f)); df["l"] = sign * df.lfc
            out[m] = df[["weed", "Cluster", "p", "q", "l"]].copy()
        return out


    CF = {(kg, s): calls_full(kg, s) for kg in ("bacteria", "fungi") for s in ("Resistant", "Permissive")}


    def universe(kg, soil, w):
        X = TT[kg]["X"]; md = TT[kg]["md"]
        idx = md[(md.soil == dc.SOILCODE[soil]) & md.species.isin([w, dc.CONTROL])].index.intersection(X.index)
        return set(X.columns[(X.loc[idx] > 0).any()])


    UNI = {(kg, s, w): universe(kg, s, w) for (kg, s), d in CF.items() for w in d["metagenomeSeq"].weed.unique()}


    def called(d, m, mode, w, mthr=None):
        df = d[m]; df = df[df.weed == w]; _, pthr, lthr = dc.THR[m]
        if m == "metagenomeSeq" and mthr is not None: lthr = mthr
        sel = (df.p < pthr) if mode == "raw" else (df.q < 0.05)
        return set(df.loc[sel & (df.l > lthr), "Cluster"])


    rows = []
    for mode, mthr in (("raw", None), ("BH", None), ("raw", 2.0), ("BH", 2.0)):
        for (kg, s), d in CF.items():
            for A, B in (("DESeq2", "edgeR"), ("DESeq2", "metagenomeSeq"), ("edgeR", "metagenomeSeq")):
                o = e = nt = ns = 0
                for w in d["metagenomeSeq"].weed.unique():
                    U = UNI[(kg, s, w)]; a = called(d, A, mode, w, mthr) & U; b = called(d, B, mode, w, mthr) & U
                    if not a or not b: continue
                    N, k = len(U), len(a & b); o += k; e += len(a) * len(b) / N; nt += 1; ns += hypergeom(N, len(a), len(b)).sf(k - 1) < 0.05
                rows.append(dict(kingdom=kg, soil=dc.SOILLABEL[s], threshold=mode + ("" if mthr is None else f", metagenomeSeq log2FC>{mthr:g}"),
                                 pair=f"{A} ∩ {B}", neighbours_tested=nt, observed=o, expected=round(e, 1), fold=round(o / e, 1) if e else np.nan,
                                 neighbours_p05=ns))
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "da_method_agreement.csv"), index=False)

    DEFS = {"D1 raw, ≥2 of 3 (v1.7-4)": ("raw", None, None, 2), "D2 raw, ≥2 of 3, common log2FC > 2 (adopted screen)": ("raw", 2.0, None, 2),
            "D3 raw, DESeq2 ∩ edgeR": ("raw", None, ("DESeq2", "edgeR"), 2), "D4 raw, all three, common log2FC > 2": ("raw", 2.0, None, 3),
            "D5 FDR, ≥2 of 3": ("BH", None, None, 2), "D6 FDR, ≥2 of 3, common log2FC > 2 (adopted core, Fig. 2)": ("BH", 2.0, None, 2)}
    fam = dict(zip(TT["bacteria"]["md"].species, TT["bacteria"]["md"].family))


    def pairs_for(dfn):
        mode, mthr, meths, kmin = dfn; out = {}
        for (kg, s), d in CF.items():
            P = set()
            for w in d["metagenomeSeq"].weed.unique():
                U = UNI[(kg, s, w)]; sets = [called(d, m, mode, w, mthr) & U for m in (meths or dc.FILES)]
                P |= {(w, c) for c in set().union(*sets) if sum(c in x for x in sets) >= kmin}
            out[(kg, s)] = P
        return out


    def convergence(P, kg, s, nperm=999, kfam=3):
        if not P: return 0, np.nan, np.nan
        byw = pd.Series([w for w, _ in P]).value_counts().sort_index()  
        def count(pp):
            df = pd.DataFrame(pp, columns=["w", "c"]); df["f"] = df.w.map(fam); return int((df.groupby("c").f.nunique() >= kfam).sum())
        obs = count(list(P)); null = []
        for _ in range(nperm):
            pp = []
            for w, k in byw.items():
                U = sorted(UNI[(kg, s, w)]); pp += [(w, c) for c in rng.choice(U, size=k, replace=False)]
            null.append(count(pp))
        null = np.array(null); return obs, float(null.mean()), float((np.sum(null >= obs) + 1) / (nperm + 1))


    def prev(kg, s, asvs, sp):
        X = TT[kg]["X"]; md = TT[kg]["md"]; idx = md[(md.soil == dc.SOILCODE[s]) & (md.species == sp)].index.intersection(X.index)
        return (X.loc[idx, list(asvs)] > 0).mean()


    rows, PAIRS = [], {}
    for name, dfn in DEFS.items():
        PP = pairs_for(dfn); PAIRS[name] = PP
        for (kg, s), P in PP.items():
            asv = {c for _, c in P}; nn = pd.Series([c for _, c in P]).value_counts() if P else pd.Series(dtype=int)
            o, e, p = convergence(P, kg, s)
            pw = prev(kg, s, asv, dc.CONTROL) if asv else pd.Series(dtype=float); pa = prev(kg, s, asv, "wheat alone") if asv else pd.Series(dtype=float)
            rows.append(dict(definition=name, kingdom=kg, soil=dc.SOILLABEL[s], calls=len(P), ASVs=len(asv),
                             genera=len({TT[kg]["tax"].Genus.get(c) for c in asv}),
                             median_neighbours_per_ASV=float(nn.median()) if len(nn) else np.nan,
                             ASVs_in_3plus_families=o, expected_3plus=round(e, 1) if e == e else np.nan, P_convergence=round(p, 3) if p == p else np.nan,
                             pct_absent_from_all_wxw_pots=round(100 * (pw == 0).mean()) if len(pw) else np.nan,
                             pct_absent_wxw_but_present_wheat_alone=round(100 * ((pw == 0) & (pa > 0)).mean()) if len(pw) else np.nan))
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "da_consensus_definitions.csv"), index=False)

    core = PAIRS["D6 FDR, ≥2 of 3, common log2FC > 2 (adopted core, Fig. 2)"]
    lfc = {(kg, s, w, c): l for (kg, s), d in CF.items() for w, c, l in zip(d["DESeq2"].weed, d["DESeq2"].Cluster, d["DESeq2"].l)}
    fam28 = {w: fam[w] for w in sorted(set(TT["bacteria"]["md"].species) - {"wheat x wheat", "wheat alone"})}
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    import si_figures as sf, bacteria_dotplot_revised as bdp, fungi_dotplot_cell as fdp
    import main_figures as mf
    mf.fig2(core, lfc, TT, fam28, os.path.join(OUT, "Fig2"))   # Times New Roman
    scr = PAIRS["D2 raw, ≥2 of 3, common log2FC > 2 (adopted screen)"]
    L = pd.DataFrame([dict(Cluster=c, Plant_Species=w, Soil=s, Kingdom=kg, Method="consensus") for (kg, s), P in scr.items() for w, c in P])
    L = L.sort_values(["Kingdom", "Soil", "Plant_Species", "Cluster"]).reset_index(drop=True)  
    L.to_csv(os.path.join(OUT, "da_consensus_screen_long.csv"), index=False)
    fi = dc.fig2_inputs(L, TT, min_combos_bacteria=5)
    with contextlib.redirect_stdout(io.StringIO()):
        fa, _ = bdp.create_combined_bacteria_plot(fi["bact"], fi["genus_b"], fi["phylum_b"], fi["family"], dc.FUNCTION_BY_GENUS,
                                                  dc.FUNCTION_PALETTE, dc.BACT_PHYLUM_COLOURS)
        fb, _ = fdp.create_combined_fungi_plot(fi["fungi"], fi["genus_f"], fi["phylum_f"], fi["family"])
        dc.match_fig2_panels(fa, fb); dc.save_fig2(fa, fb, os.path.join(OUT, "FigS17_genus_screen"))
    plt.close("all")
    # ---------------------------------------------------------------- Table S7 workbook (v2.3)
    def tier_calls(mode):
        rows = []
        for (kg, s), d in CF.items():
            for w in d["metagenomeSeq"].weed.unique():
                U = UNI[(kg, s, w)]; sets = {m: called(d, m, mode, w, 2.0) & U for m in dc.FILES}
                for c in set().union(*sets.values()):
                    ms = [m for m in dc.FILES if c in sets[m]]
                    if len(ms) >= 2:
                        rows.append(dict(kingdom=kg, soil=dc.SOILLABEL[s], neighbour=w, ASV=c, methods=", ".join(ms)))
        return pd.DataFrame(rows)


    SCR, FDR = tier_calls("raw"), tier_calls("BH")
    assert {(k, w, c) for k, w, c in zip(FDR.kingdom, FDR.neighbour, FDR.ASV)} == {(k, w, c) for (k, s), P in core.items() for w, c in P}
    fdr_keys = set(zip(FDR.kingdom, FDR.soil, FDR.neighbour, FDR.ASV))
    SCR["FDR_supported"] = ["yes" if k in fdr_keys else "no" for k in zip(SCR.kingdom, SCR.soil, SCR.neighbour, SCR.ASV)]


    def per_asv(kg):
        g = SCR[SCR.kingdom == kg]; tx = TT[kg]["tax"]; out = []
        for c, h in g.groupby("ASV"):
            hf = h[h.FDR_supported == "yes"]
            out.append(dict(ASV=c.replace("Cluster_", "ASV_"), Phylum=tx.Phylum.get(c), Genus=tx.Genus.get(c), Species=tx.Species.get(c),
                            Soil=", ".join(sorted(h.soil.unique())), n_neighbours=h.neighbour.nunique(),
                            n_families=h.neighbour.map(fam).nunique(),
                            Method_combinations="; ".join(sorted(h.methods.unique())),
                            Neighbours="; ".join(f"{s}: " + ", ".join(sorted(x.neighbour)) for s, x in h.groupby("soil")),
                            In_Fig2="yes" if len(hf) else "no",
                            Neighbours_FDR="; ".join(f"{s}: " + ", ".join(sorted(x.neighbour)) for s, x in hf.groupby("soil"))))
        return pd.DataFrame(out).sort_values(["Phylum", "Genus", "ASV"])


    ov = pd.concat([SCR.assign(tier="unadjusted P (screen)"), FDR.assign(tier="FDR q < 0.05 (Fig. 2)")])
    ov = ov.groupby(["tier", "kingdom", "soil", "methods"]).size().unstack("methods", fill_value=0).reset_index()
    ov["calls"] = ov[[c for c in ov.columns if "," in c]].sum(1)
    ag = pd.read_csv(os.path.join(OUT, "da_method_agreement.csv"))
    ag = ag[ag.threshold.isin(["raw, metagenomeSeq log2FC>2", "BH, metagenomeSeq log2FC>2"])].copy()
    ag["threshold"] = ag.threshold.map({"raw, metagenomeSeq log2FC>2": "unadjusted P (screen)", "BH, metagenomeSeq log2FC>2": "FDR q < 0.05 (Fig. 2)"})
    readme = pd.DataFrame({"Table S7": [
        "Neighbour-enriched ASVs in the wheat root endosphere. An ASV × neighbour × soil call requires enrichment under the neighbour relative to "
        "wheat × wheat by at least two of DESeq2, edgeR and metagenomeSeq, each with log2 fold change > 2.",
        "Fig. 2 (adjusted P): DESeq2 padj and edgeR FDR (Benjamini-Hochberg), metagenomeSeq Bonferroni-adjusted P, each < 0.05. Screen (unadjusted): DESeq2 P < 0.05, edgeR P < 0.01, "
        "metagenomeSeq adjusted P < 0.05. Every Fig. 2 call is also a screen call.",
        "The fold-change sign of every export was checked against log2(neighbour/control) computed from the abundance table.",
        "Sheets: bacteria, fungi (one row per ASV; In_Fig2 = passes at FDR); calls (one row per ASV × neighbour × soil); method_overlap "
        "(calls by method combination, both tiers); method_agreement (observed vs chance overlap of each pair of methods; Methods S3)."]})
    xl = os.path.join(OUT, "Table_S7_enriched_ASVs.xlsx")
    with pd.ExcelWriter(xl) as w:
        readme.to_excel(w, sheet_name="README", index=False)
        per_asv("bacteria").to_excel(w, sheet_name="bacteria", index=False); per_asv("fungi").to_excel(w, sheet_name="fungi", index=False)
        SCR.assign(ASV=SCR.ASV.str.replace("Cluster_", "ASV_")).sort_values(["kingdom", "soil", "neighbour", "ASV"]).to_excel(w, sheet_name="calls", index=False)
        ov.to_excel(w, sheet_name="method_overlap", index=False)
        ag[["threshold", "kingdom", "soil", "pair", "neighbours_tested", "observed", "expected", "fold", "neighbours_p05"]].to_excel(w, sheet_name="method_agreement", index=False)
    print("Table S7:", xl)

    print("done:", OUT)


In [ ]:
DA_OUT = os.path.join(OUT_DIR, "differential_abundance"); os.makedirs(DA_OUT, exist_ok=True)
da_core_check(MATRIX_DIR, DA_OUT, DA_ROOTS)   # [I2] was: python da_core_check.py
conv = pd.read_csv(os.path.join(DA_OUT, "da_consensus_definitions.csv"))
conv[conv.definition.str.startswith("D6")][["kingdom", "soil", "calls", "ASVs", "ASVs_in_3plus_families", "expected_3plus", "P_convergence"]]

Table S7: /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/results/differential_abundance/Table_S7_enriched_ASVs.xlsx
done: /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/results/differential_abundance


,kingdom,soil,calls,ASVs,ASVs_in_3plus_families,expected_3plus,P_convergence
20,bacteria,cereal-dominated,143,15,15,0.3,0.001
21,bacteria,mixed-system,70,13,7,0.0,0.001
22,fungi,cereal-dominated,12,1,1,0.0,0.001
23,fungi,mixed-system,71,9,6,0.2,0.001


### Methods S3, Table S3: control-size check (all weeds pooled against wheat × wheat; each call against the other weeds; each call against wheat alone)

In [ ]:
CS_OUT = os.path.join(SUPP, "control_size"); os.makedirs(CS_OUT, exist_ok=True)
fig2v = pd.read_csv(os.path.join(DA_OUT, "Fig2_values.csv"))
_spec = os.path.join(CS_OUT, "fig2_neighbour_vs_other_weeds.csv"); _k = ["kingdom", "soil", "neighbour", "ASV"]
_fresh = os.path.exists(os.path.join(CS_OUT, "vs_wheat_alone_all.csv")) and os.path.exists(_spec) and set(map(tuple, pd.read_csv(_spec)[_k].values)) == set(map(tuple, fig2v[_k].values))
if not _fresh:                                                    # c. 5-20 min; rerun whenever the Fig. 2 calls change
    csz.run(MATRIX_DIR, os.path.join(DA_OUT, "Fig2_values.csv"), CS_OUT, RSCRIPT)
pooled = pd.read_csv(os.path.join(CS_OUT, "pooled_weeds_vs_wxw.csv")); spec = pd.read_csv(os.path.join(CS_OUT, "fig2_neighbour_vs_other_weeds.csv"))
alone = pd.read_csv(os.path.join(CS_OUT, "vs_wheat_alone_all.csv"))
calls = csz.classify(fig2v, pooled, spec, alone)
csz.table_s3(os.path.join(DA_OUT, "Table_S7_enriched_ASVs.xlsx"), calls, pooled, os.path.join(TAB, "Table_S3_enriched_ASVs.xlsx"))
csz.summary(calls)

{'calls': 296,
 'ASVs': 38,
 'ASVs_called_all_weeds': 1,
 'calls_specific_both': 0,
 'calls_specific_one': 26,
 'calls_vs_alone': 14,
 'ASVs_vs_alone': 6,
 'share_positive_vs_alone': 0.82}

### Results 3.2.2: overlap of the Fig. 2 ASVs with the TITAN2 indicators

`core_vs_indicators()`: overlap of the Fig. 2 ASVs with the TITAN2 indicators.   <!-- [I3] formerly core_vs_indicators.py -->

In [ ]:
def core_vs_indicators(DF, S5, CORE, OUT):
    """Overlap of the FDR differential-abundance core (Fig. 2) with the TITAN2 indicators of wheat biomass (z+ and z-).
    Universe = the eligible set of each kingdom x soil (present in >= 10% of pots with biomass, mean relative abundance >= 1e-4), as in
    the ranking test. Expected overlap = n_core * n_set / N; P = hypergeometric upper tail; pooled P = exact distribution of the summed
    count over the four kingdom x soil cells (convolution), by exact convolution."""
    import sys
    import numpy as np, pandas as pd
    from scipy.stats import hypergeom
    s5, core = pd.read_csv(S5), pd.read_csv(CORE)
    code = {"Cereal-dominated (AF193)": "resistant", "Mixed-system (AF190)": "Permissive"}
    rows, conv, obs = [], {k: np.array([1.0]) for k in ("z+", "z-")}, {k: 0 for k in ("z+", "z-")}
    for kg, n in (("bacteria", 1314), ("fungi", 647)):
        X0 = pd.read_csv(f"{DF}/otu_{kg}_table_R_{n}ASVs.csv", sep="\t", index_col=0).T.astype(float)
        md = pd.read_csv(f"{DF}/metadata_{kg}_{n}ASVs.csv").set_index("sample")
        for so, sc in code.items():
            m = md.loc[md.index.intersection(X0.index)]; m = m[(m.soil == sc) & m.Dry_wheat_mass.notna()]
            X = X0.loc[m.index]; X = X.div(X.sum(1), axis=0)
            elig = set(X.columns[((X > 0).mean() >= 0.10) & (X.mean() >= 1e-4)]); N = len(elig)
            c = set(core[(core.kingdom == kg) & (core.soil == so)].ASV) & elig
            g = s5[(s5.kingdom == kg) & (s5.soil == so)]
            sets = {"z+": set(g[g.TITAN2_direction == "z+"].ASV) & elig,
                    "z-": set(g[g.TITAN2_direction == "z-"].ASV) & elig}
            row = dict(kingdom=kg, soil=so, eligible=N, core_in_eligible=len(c))
            for k, A in sets.items():
                o = len(c & A); obs[k] += o
                conv[k] = np.convolve(conv[k], hypergeom(N, len(A), len(c)).pmf(np.arange(len(c) + 1)))
                row.update({f"{k}_n": len(A), f"{k}_shared": o, f"{k}_expected": round(len(c) * len(A) / N, 2),
                            f"{k}_P": round(hypergeom(N, len(A), len(c)).sf(o - 1), 3) if o else 1.0,
                            f"{k}_ASVs": ", ".join(sorted(x.replace("Cluster_", "ASV_") for x in c & A))})
            rows.append(row)
    T = pd.DataFrame(rows)
    T.loc[len(T)] = dict(kingdom="all", soil="pooled", **{f"{k}_shared": obs[k] for k in obs},
                         **{f"{k}_expected": round(T[f"{k}_expected"].sum(), 2) for k in obs}, **{f"{k}_P": round(conv[k][obs[k]:].sum(), 3) for k in obs})
    T.to_csv(OUT, index=False)
    print(T.to_string(index=False))


In [ ]:
core_vs_indicators(MATRIX_DIR, os.path.join(RESULTS, "Table_S7_TITAN2_indicators.csv"),   # (table-s7-titan2)
                    os.path.join(DA_OUT, "Fig2_values.csv"), os.path.join(TAB, "fig2_vs_indicators.csv"))  
pd.read_csv(os.path.join(TAB, "fig2_vs_indicators.csv"))[["kingdom", "soil", "z+_shared", "z+_expected", "z+_P", "z+_ASVs"]]

 kingdom                     soil  eligible  core_in_eligible  z+_n  z+_shared  z+_expected  z+_P           z+_ASVs  z-_n  z-_shared  z-_expected  z-_P z-_ASVs
bacteria Cereal-dominated (AF193)     808.0              15.0  20.0          0         0.37 1.000                    15.0          0         0.28   1.0        
bacteria     Mixed-system (AF190)     903.0              13.0  33.0          2         0.48 0.079 ASV_1201, ASV_241  23.0          0         0.33   1.0        
   fungi Cereal-dominated (AF193)     124.0               1.0   4.0          0         0.03 1.000                     1.0          0         0.01   1.0        
   fungi     Mixed-system (AF190)     153.0               7.0  11.0          0         0.50 1.000                     1.0          0         0.05   1.0        
     all                   pooled       NaN               NaN   NaN          2         1.38 0.406               NaN   NaN          0         0.67   1.0     NaN


,kingdom,soil,z+_shared,z+_expected,z+_P,z+_ASVs
0,bacteria,Cereal-dominated (AF193),0,0.37,1.000,NaN
1,bacteria,Mixed-system (AF190),2,0.48,0.079,"ASV_1201, ASV_241"
2,fungi,Cereal-dominated (AF193),0,0.03,1.000,NaN
3,fungi,Mixed-system (AF190),0,0.50,1.000,NaN
4,all,pooled,2,1.38,0.406,NaN


## 3.2.1 Wheat biomass

### Fig. 3 (between-soil tests, BH across 30 treatments); log-scale ANOVA; legume test; Table S4

In [ ]:
META_B = os.path.join(MATRIX_DIR, "metadata_bacteria_1314ASVs.csv")
m = bt.load(META_B)
fig3, soil_dependent = mf.fig3(MATRIX_DIR, os.path.join(FIG, "Fig3"))
aov = bt.anova_tables(m); aov.to_csv(os.path.join(TAB, "biomass_anova.csv"), index=False)
bt.scale_check(m).to_csv(os.path.join(TAB, "biomass_scale_check.csv"), index=False)   # Methods 2.7 and S3: why log scale
between = bt.between_soils(m); between.to_csv(os.path.join(TAB, "biomass_between_soils.csv"), index=False)
leg = bt.legume_test(m); leg.to_csv(os.path.join(TAB, "biomass_legumes.csv"), index=False)
t4 = bt.table_s3(m, os.path.join(TAB, "Table_S4_biomass_contrasts.xlsx"))     # Table S4 in the SI
print("asterisks:", soil_dependent, "| above W×W (q < 0.05):", int((t4.q_vs_wxw < 0.05).sum()), "| above wheat alone:", int((t4.delta_vs_alone_g > 0).sum()))
aov[aov.scale == "log"].round(3)

asterisks: ['Achillea millefolium', 'Lolium arundinaceum'] | above W×W (q < 0.05): 45 | above wheat alone: 0


,set,scale,term,df,df_resid,F,P,shapiro_W,brown_forsythe_P
3,all treatments,log,soil,1,388,0.316,0.574,0.962,0.001
4,all treatments,log,species,29,388,17.167,0.000,0.962,0.001
5,all treatments,log,soil:species,29,388,2.857,0.000,0.962,0.001
9,weeds only,log,soil,1,361,0.208,0.649,0.962,0.003
10,weeds only,log,species,27,361,12.517,0.000,0.962,0.003
11,weeds only,log,soil:species,27,361,2.909,0.000,0.962,0.003


## 3.2.2 Microbiota shifts and wheat growth

### Fig. 4 (species-centroid db-RDA on species-mean biomass; tests from qmd 3.1.1); Table S6 (sensitivity of the Fig. 4 tests); Table S7 (TITAN2 indicators, exploratory)

In [ ]:
fig4 = mf.fig4(os.path.join(REV, "fig4_scores.csv"), os.path.join(REV, "fig4_direct_test.csv"), os.path.join(FIG, "Fig4"))
s5 = pd.read_csv(os.path.join(RESULTS, "Table_S7_TITAN2_indicators.csv"))   
t6 = s5[s5.TITAN2_direction.isin(["z+", "z-"])][["kingdom", "soil", "ASV", "Genus", "TITAN2_direction", "spearman_rho_biomass"]].copy()
t6["ASV"] = t6.ASV.str.replace("Cluster_", "ASV_")
t6["in_Fig2"] = [(k, a) in set(zip(fig2v.kingdom, fig2v.ASV.str.replace("Cluster_", "ASV_"))) for k, a in zip(t6.kingdom, t6.ASV)]
t6["responds_in_both_soils"] = t6.groupby(["kingdom", "ASV"]).soil.transform("nunique") > 1
t6.to_excel(os.path.join(TAB, "Table_S7_TITAN2_indicators.xlsx"), index=False)
t6s = pd.read_csv(os.path.join(REV, "fig4_direct_test.csv"))[["kingdom", "soil", "R2_pct", "P", "P_given_monocot", "P_given_monocot_legume", "P_given_Btot", "P_without_legumes"]]
t6s.round(3).to_csv(os.path.join(TAB, "Table_S6_fig4_sensitivity.csv"), index=False)
print(t6.groupby(["kingdom", "soil", "TITAN2_direction"]).size())
pd.read_csv(os.path.join(REV, "fig4_direct_test.csv")).round(4)

kingdom   soil                      TITAN2_direction
bacteria  Cereal-dominated (AF193)  z+                  24
                                    z-                  15
          Mixed-system (AF190)      z+                  36
                                    z-                  25
fungi     Cereal-dominated (AF193)  z+                   6
                                    z-                   1
          Mixed-system (AF190)      z+                  14
                                    z-                   1
dtype: int64


,kingdom,soil,n_species,R2_pct,F,P,P_given_monocot,P_given_monocot_legume,P_given_Btot,R2_adj_pct,R2_without_legumes_pct,P_without_legumes,r_biomass_monocot,q_BH
0,bacteria,Cereal-dominated (AF193),28,5.3075,1.4573,0.0100,0.0264,0.0067,0.0131,1.6655,6.8668,0.0031,-0.2169,0.0400
1,bacteria,Mixed-system (AF190),28,4.5380,1.2360,0.0593,0.0332,0.0152,0.1169,0.8664,5.4039,0.0560,-0.0023,0.1186
2,fungi,Cereal-dominated (AF193),28,4.1880,1.1365,0.2554,0.5257,0.0781,0.3631,0.5029,6.3958,0.0473,-0.2693,0.3405
3,fungi,Mixed-system (AF190),28,3.7122,1.0024,0.4498,0.4232,0.6171,0.4419,0.0088,4.2644,0.4947,-0.0881,0.4498


## 3.3 Weed traits and phylogeny

### Table 2

In [ ]:
comp = pd.read_csv(os.path.join(TRAITS, "trait_comp.csv")); joint = pd.read_csv(os.path.join(TRAITS, "trait_joint.csv"))
mant = pd.read_csv(os.path.join(TRAITS, "trait_mantel.csv"))
star = lambda p: "***" if p <= 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
cells_ = [(k, s) for k in ("bacteria", "fungi") for s in ("Cereal-dominated (AF193)", "Mixed-system (AF190)")]
lab = [("Phylogeny", "Distance to wheat", "Distance"), ("", "Monocot vs eudicot", "Monocot"), ("", "Segetal status", "Segetal"),
       ("Habitat", "Ellenberg N", "Nitrogen"), ("", "Ellenberg L", "Light"), ("Root traits", "SRL", "SRL"), ("", "RDMC", "RDMC"), ("", "Root:shoot", "RootShoot")]
rows = []
for c1, c2, k in lab:
    rows.append([c1, c2] + [f"{r.F:.2f}{star(r.P_species)}" for r in [comp[(comp.kingdom == a) & (comp.soil == b) & (comp.trait == k)].iloc[0] for a, b in cells_]])
J = joint.set_index(["kingdom", "soil"]); MT = mant.set_index(["kingdom", "soil"])
rows.append(["Joint model", "R² of seven traits (%)"] + [f"{100 * J.loc[c].R2_total:.1f} (P = {J.loc[c].P_species:.2f})" for c in cells_])
rows.append(["", "R² of species identity (%)"] + [f"{100 * J.loc[c].R2_species:.1f}" for c in cells_])
rows.append(["Phylogenetic signal", "Mantel r"] + [f"{MT.loc[c].mantel_r:.2f}{star(MT.loc[c].P)}" for c in cells_])
t2 = pd.DataFrame(rows, columns=["Trait category", "Variable", "Bacteria AF193", "Bacteria AF190", "Fungi AF193", "Fungi AF190"])
t2.to_excel(os.path.join(TAB, "Table_2_traits.xlsx"), index=False); t2

,Trait category,Variable,Bacteria AF193,Bacteria AF190,Fungi AF193,Fungi AF190
0,Phylogeny,Distance to wheat,2.14*,2.72**,2.99***,2.33**
1,,Monocot vs eudicot,1.96*,2.55***,3.00**,2.31**
2,,Segetal status,1.34,1.65,1.07,1.07
3,Habitat,Ellenberg N,1.64,2.27**,1.41,1.49
4,,Ellenberg L,1.58,1.45,1.75,1.39
5,Root traits,SRL,1.34,1.56,1.52,1.49
6,,RDMC,1.22,1.37,1.07,1.39
7,,Root:shoot,1.31,1.46,1.74,1.53
8,Joint model,R² of seven traits (%),4.8 (P = 0.76),5.6 (P = 0.11),5.2 (P = 0.16),4.7 (P = 0.24)
9,,R² of species identity (%),18.6,19.3,17.8,16.5


### Table 2 robustness (Notes S3): 

In [ ]:
def table2_robustness(DF, TRAITS, DIST, OUT):
    """Robustness of Table 2 (Notes S6): the Distance term under other dissimilarities"""
    import sys, os
    import numpy as np, pandas as pd
    from scipy.spatial.distance import pdist, squareform

    B = 999
    rng = np.random.default_rng(7)

    def gower(D, lingoes=True):
        n = len(D); J = np.eye(n) - 1 / n; G = J @ (-0.5 * D ** 2) @ J
        if lingoes:
            ev = np.linalg.eigvalsh(G)
            if ev.min() < -1e-10 * abs(ev).max():
                D2 = np.sqrt(D ** 2 + 2 * (-ev.min())); np.fill_diagonal(D2, 0); G = J @ (-0.5 * D2 ** 2) @ J
        return G

    def Fstat(x, G):
        X = np.column_stack([np.ones(len(G)), x]); Q, _ = np.linalg.qr(X)
        s = np.sum(Q * (G @ Q)); tot = np.trace(G); p = X.shape[1] - 1
        return (s / p) / ((tot - s) / (len(G) - p - 1))

    def species_perm(sp, val, G):
        u = pd.Series(val, index=sp).groupby(level=0).first()
        x = pd.Series(sp).map(u).values.astype(float); Fo = Fstat(x, G)
        Fp = np.array([Fstat(pd.Series(sp).map(dict(zip(u.index, rng.permutation(u.values)))).values.astype(float), G) for _ in range(B)])
        return f"({Fo:.3f}, {(np.sum(Fp >= Fo) + 1) / (B + 1):.3f})"

    ST = pd.read_csv(TRAITS).set_index("species"); cd = pd.read_csv(DIST).set_index("species")
    mono = {s: int(cd.loc[s, "recomputed"] < 200) for s in cd.index}
    TR = {"Distance": cd.recomputed, "Segetal": ST.Segetal, "Nitrogen": ST.EllenN, "Light": ST.EllenL,
          "SRL": ST.SRL, "RDMC": ST.RDMC, "RootShoot": ST.RootShoot}
    rows, cen_rows = [], []
    for kg, tag in (("bacteria", "1314"), ("fungi", "647")):
        otu = pd.read_csv(os.path.join(DF, f"otu_{kg}_table_R_{tag}ASVs.csv"), sep="\t", index_col=0).T.astype(float)
        md = pd.read_csv(os.path.join(DF, f"metadata_{kg}_{tag}ASVs.csv")).set_index("sample")
        for code, sl in (("resistant", "cereal"), ("Permissive", "mixed")):
            m = md.loc[md.index.intersection(otu.index)]
            m = m[(m.soil == code) & ~m.species.isin(["wheat alone", "wheat x wheat"])]
            X = otu.loc[m.index].values; sp = m.species.values
            Dbc = squareform(pdist(X, "braycurtis"))
            H = np.sqrt(X / X.sum(1, keepdims=True))
            dist = pd.Series(sp).map(cd.recomputed).values
            nj = ~pd.Series(sp).isin(["Juncus bufonius"]).values
            r = dict(kingdom=kg, soil=sl, n=len(sp))
            r["Distance BC+Lingoes"] = species_perm(sp, dist, gower(Dbc))
            r["Distance BC no correction"] = species_perm(sp, dist, gower(Dbc, False))
            r["Distance Hellinger"] = species_perm(sp, dist, gower(squareform(pdist(H, "euclidean")), False))
            r["Distance Jaccard (presence)"] = species_perm(sp, dist, gower(squareform(pdist(X > 0, "jaccard"))))
            r["Monocot vs eudicot (binary)"] = species_perm(sp, pd.Series(sp).map(mono).values, gower(Dbc))
            r["Distance, Juncus removed"] = species_perm(sp[nj], dist[nj], gower(Dbc[np.ix_(nj, nj)]))
            rows.append(r)
            Xr = otu.loc[m.index]; Xr = Xr.div(Xr.sum(1), axis=0); C = Xr.groupby(m.species.values).mean()
            c = dict(kingdom=kg, soil=sl)
            for t, s in TR.items():
                v = s.reindex(C.index).astype(float); ok = v.notna().values
                G = gower(squareform(pdist(C.values[ok], "braycurtis"))); x = v.values[ok]
                Fo = Fstat(x, G); Fp = np.array([Fstat(rng.permutation(x), G) for _ in range(B)])
                c[t] = f"{Fo:.2f} ({(np.sum(Fp >= Fo) + 1) / (B + 1):.3f})"
            cen_rows.append(c)
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "table3_robustness.csv"), index=False)
    pd.DataFrame(cen_rows).to_csv(os.path.join(OUT, "table3_species_centroid_check.csv"), index=False)
    print(pd.DataFrame(rows).to_string(index=False)); print(pd.DataFrame(cen_rows).to_string(index=False))


In [ ]:
T2R = os.path.join(TAB, "table2_robustness"); os.makedirs(T2R, exist_ok=True)
if not os.path.exists(os.path.join(T2R, "table3_robustness.csv")):          # 
    table2_robustness(MATRIX_DIR, os.path.join(CODE_DIR, "species_traits.csv"),
                      os.path.join(TRAITS, "check_phylogenetic_distance.csv"), T2R)
pd.read_csv(os.path.join(T2R, "table3_robustness.csv"))

 kingdom   soil   n Distance BC+Lingoes Distance BC no correction Distance Hellinger Distance Jaccard (presence) Monocot vs eudicot (binary) Distance, Juncus removed
bacteria cereal 209      (2.143, 0.018)            (2.270, 0.022)     (2.202, 0.016)              (1.986, 0.043)              (1.959, 0.042)           (2.184, 0.014)
bacteria  mixed 215      (2.723, 0.001)            (2.857, 0.001)     (2.668, 0.004)              (2.313, 0.007)              (2.549, 0.002)           (2.744, 0.001)
   fungi cereal 209      (2.992, 0.001)            (4.417, 0.002)     (4.388, 0.002)              (2.847, 0.002)              (3.001, 0.001)           (2.984, 0.002)
   fungi  mixed 210      (2.334, 0.008)            (3.419, 0.007)     (3.265, 0.003)              (2.392, 0.004)              (2.308, 0.007)           (2.347, 0.006)
 kingdom   soil     Distance      Segetal     Nitrogen        Light          SRL         RDMC    RootShoot
bacteria cereal 1.26 (0.052) 0.92 (0.649) 1.16 (0.156) 1.04 (0.

,kingdom,soil,n,Distance BC+Lingoes,Distance BC no correction,Distance Hellinger,Distance Jaccard (presence),Monocot vs eudicot (binary),"Distance, Juncus removed"
0,bacteria,cereal,209,"(2.143, 0.018)","(2.270, 0.022)","(2.202, 0.016)","(1.986, 0.043)","(1.959, 0.042)","(2.184, 0.014)"
1,bacteria,mixed,215,"(2.723, 0.001)","(2.857, 0.001)","(2.668, 0.004)","(2.313, 0.007)","(2.549, 0.002)","(2.744, 0.001)"
2,fungi,cereal,209,"(2.992, 0.001)","(4.417, 0.002)","(4.388, 0.002)","(2.847, 0.002)","(3.001, 0.001)","(2.984, 0.002)"
3,fungi,mixed,210,"(2.334, 0.008)","(3.419, 0.007)","(3.265, 0.003)","(2.392, 0.004)","(2.308, 0.007)","(2.347, 0.006)"


## Display items

Every item under its manuscript name in `display_items/`. Figs S1–S3 are drawings or images supplied with the manuscript
(design, soil characterisation, rarefaction); Fig. S4 and Tables 1 and S5 are written by `Paper_analyses.qmd`.

In [ ]:
ITEMS = {
 "Fig_1": os.path.join(FIG, "Fig1.png"), "Fig_2": os.path.join(DA_OUT, "Fig2.png"), "Fig_3": os.path.join(FIG, "Fig3.png"), "Fig_4": os.path.join(FIG, "Fig4.png"),
 "Table_1": os.path.join(RESULTS, "Table1_variance_partition.csv"), "Table_2": os.path.join(TAB, "Table_2_traits.xlsx"),
 "Fig_S4": os.path.join(RESULTS, "FigS4_permdisp.png"), "Fig_S5": os.path.join(FIG, "FigS5_alpha_diversity.png"),
 "Fig_S6": os.path.join(FIG, "FigS6_glomeromycota.png"), "Fig_S7": os.path.join(FIG, "FigS7_assembly_bacteria.png"),
 "Table_S3": os.path.join(TAB, "Table_S3_enriched_ASVs.xlsx"), "Table_S4": os.path.join(TAB, "Table_S4_biomass_contrasts.xlsx"),
 "Table_S5": os.path.join(RESULTS, "Table_S6_biomass_ancova.csv"), "Table_S6": os.path.join(TAB, "Table_S6_fig4_sensitivity.csv"), "Table_S7": os.path.join(TAB, "Table_S7_TITAN2_indicators.xlsx"),
 "Graphical_abstract": os.path.join(FIG, "graphical_abstract.png")}
DI = os.path.join(OUT_DIR, "display_items"); os.makedirs(DI, exist_ok=True)
status = {}
for name, src in ITEMS.items():
    stem, ext = os.path.splitext(src)
    for e in ([ext] + [x for x in (".pdf", ".svg") if ext == ".png"]):
        if os.path.exists(stem + e): shutil.copy(stem + e, os.path.join(DI, name + e))
    status[name] = os.path.exists(src)
missing = [k for k, v in status.items() if not v]
print(f"{sum(status.values())} of {len(status)} display items written to {DI}; missing: {missing}")

16 of 16 display items written to /Users/mathieuvanhove/Desktop/Work/DeepImpact/Weeds_wheat/submission_v2.1/results/display_items; missing: []
